# AI Detection

# Notebook overview

This notebook documents the **multi-detector AI-detection workflow** used in the study. It includes the main detector runs, specialized runs for the testing-sample subsets, retry scripts for failed Pangram calls, reconstruction utilities for log files, and a final consolidation step that merges results from multiple branches of the project.

**Main workflow**
1. Run the main written-assignment corpus through multiple AI detectors.
2. Run the testing-sample subsets through the same or revised detector logic.
3. Repair failed Pangram rows without rerunning the entire experiment.
4. Reconstruct or combine detector outputs when partial logs or separate result files exist.
5. Produce consolidated outputs for later statistical analysis.

## Main written-assignment detection run

### Code block 1 — Run the main corpus through four detectors

This block processes the main **AI Trim** corpus using GPTZero, Pangram, Sapling, and Isgen. It is the core multi-detector written-text pipeline and writes one result row per file while skipping work that has already been completed.

In [ ]:
# Install dependencies when running in Colab.
!pip -q install requests pandas

import json
import os
import re
import time

import pandas as pd
import requests

# Configure paths and credentials through environment variables or Colab secrets.
AI_TRIM_ROOT = os.environ["AI_TRIM_ROOT"]
EXPERIMENT_FOLDER = os.environ["EXPERIMENT_FOLDER"]
OUTPUT_CSV = os.environ["OUTPUT_CSV"]
FAILED_CSV = os.environ["FAILED_CSV"]

GPTZERO_API_KEY = os.environ["GPTZERO_API_KEY"]
PANGRAM_API_KEY = os.environ["PANGRAM_API_KEY"]
SAPLING_API_KEY = os.environ["SAPLING_API_KEY"]
ISGEN_RAPIDAPI_KEY = os.environ["ISGEN_RAPIDAPI_KEY"]

MAX_TEXT_CHARS = 50_000
MAX_RETRIES = 2
WAIT_SECONDS_BETWEEN_RETRIES = 12
REQUEST_TIMEOUT = 180


def ensure_folder_exists(path):
    folder = os.path.dirname(path)
    if folder:
        os.makedirs(folder, exist_ok=True)


def list_target_files(root_folder):
    target_files = []
    skipped_pdfs = []
    for root, _, files in os.walk(root_folder):
        for filename in files:
            full_path = os.path.join(root, filename)
            if filename.lower().endswith(".txt"):
                target_files.append(full_path)
            elif filename.lower().endswith(".pdf"):
                skipped_pdfs.append(full_path)
    return sorted(target_files), sorted(skipped_pdfs)


def get_subfolder_name(file_path, root_folder):
    rel_dir = os.path.relpath(os.path.dirname(file_path), root_folder)
    return "(root)" if rel_dir == "." else os.path.basename(os.path.dirname(file_path))


def count_words(text):
    return len(re.findall(r"\S+", text or ""))


def detect_language_simple(text):
    if not text:
        return "en"
    arabic_chars = re.findall(r"[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF]", text)
    return "ar" if len(arabic_chars) / len(text) > 0.03 else "en"


def clean_text(text):
    if not text:
        return ""
    text = text.replace("\x00", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def truncate_text(text, max_chars=MAX_TEXT_CHARS):
    if len(text) <= max_chars:
        return text

    cut = text[:max_chars]
    last_paragraph = cut.rfind("\n\n")
    if last_paragraph > max_chars * 0.6:
        return cut[:last_paragraph].strip()

    last_boundary = max(
        cut.rfind(". "), cut.rfind("! "), cut.rfind("? "),
        cut.rfind("۔ "), cut.rfind("\n"),
    )
    if last_boundary > max_chars * 0.6:
        return cut[:last_boundary + 1].strip()
    return cut.strip()


def extract_text_from_txt(file_path):
    for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            with open(file_path, "r", encoding=encoding, errors="strict") as file:
                return file.read()
        except (UnicodeDecodeError, OSError):
            continue

    with open(file_path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()


def request_with_retries(call_name, func, failed_calls, file_path):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return func()
        except Exception as error:
            last_error = str(error)
            print(f"    {call_name}: attempt {attempt}/{MAX_RETRIES} failed -> {error}")
            if attempt < MAX_RETRIES:
                time.sleep(WAIT_SECONDS_BETWEEN_RETRIES)

    failed_calls.append({
        "file_path": file_path,
        "tool": call_name,
        "error": last_error or "Unknown error",
    })
    return None


def get_experiment_file_names(experiment_folder):
    if not os.path.exists(experiment_folder):
        return set()
    return {
        name
        for name in os.listdir(experiment_folder)
        if os.path.isfile(os.path.join(experiment_folder, name))
        and name.lower().endswith(".txt")
    }


def get_already_processed_paths_and_names(csv_path):
    if not os.path.exists(csv_path):
        return set(), set()

    try:
        df = pd.read_csv(csv_path)
    except Exception as error:
        print(f"Could not read existing CSV for skip logic: {error}")
        return set(), set()

    paths = set(df["file_path"].dropna().astype(str)) if "file_path" in df else set()
    names = set(df["file_name"].dropna().astype(str)) if "file_name" in df else set()
    return paths, names


def call_sapling(text, api_key):
    response = requests.post(
        "https://api.sapling.ai/api/v1/aidetect",
        json={"key": api_key, "text": text},
        timeout=REQUEST_TIMEOUT,
    )
    if not response.ok:
        raise RuntimeError(f"Sapling HTTP {response.status_code}: {response.text[:500]}")

    data = response.json()
    return {
        "sapling_score": data.get("score"),
        "sapling_text_len_chars": len(data.get("text", "") or ""),
        "sapling_num_sentence_scores": len(data.get("sentence_scores", []) or []),
        "sapling_raw_json": json.dumps(data, ensure_ascii=False),
    }


def call_isgen(text, lang, api_key):
    response = requests.post(
        "https://ai-detection4.p.rapidapi.com/v1/ai-detection-rapid-api",
        json={"text": text, "lang": lang},
        headers={
            "x-rapidapi-key": api_key,
            "x-rapidapi-host": "ai-detection4.p.rapidapi.com",
            "Content-Type": "application/json",
        },
        timeout=REQUEST_TIMEOUT,
    )
    if not response.ok:
        raise RuntimeError(f"Isgen HTTP {response.status_code}: {response.text[:500]}")

    data = response.json()
    summary = data.get("summary", {}) or {}
    return {
        "isgen_ai_score": data.get("ai_score"),
        "isgen_lang": data.get("lang"),
        "isgen_total_num_words": data.get("total_num_words"),
        "isgen_summary_ai": summary.get("ai"),
        "isgen_summary_human": summary.get("human"),
        "isgen_summary_mixed": summary.get("mixed"),
        "isgen_num_paragraphs": len(data.get("paragraphs", []) or []),
        "isgen_raw_json": json.dumps(data, ensure_ascii=False),
    }


def call_pangram(text, api_key):
    response = requests.post(
        "https://text.api.pangram.com/v3",
        headers={"Content-Type": "application/json", "x-api-key": api_key},
        json={"text": text, "public_dashboard_link": False},
        timeout=REQUEST_TIMEOUT,
    )
    if not response.ok:
        raise RuntimeError(f"Pangram HTTP {response.status_code}: {response.text[:500]}")

    data = response.json()
    return {
        "pangram_prediction_short": data.get("prediction_short"),
        "pangram_prediction": data.get("prediction"),
        "pangram_headline": data.get("headline"),
        "pangram_fraction_ai": data.get("fraction_ai"),
        "pangram_fraction_ai_assisted": data.get("fraction_ai_assisted"),
        "pangram_fraction_human": data.get("fraction_human"),
        "pangram_num_ai_segments": data.get("num_ai_segments"),
        "pangram_num_ai_assisted_segments": data.get("num_ai_assisted_segments"),
        "pangram_num_human_segments": data.get("num_human_segments"),
        "pangram_num_windows": len(data.get("windows", []) or []),
        "pangram_raw_json": json.dumps(data, ensure_ascii=False),
    }


def call_gptzero_text(text, api_key):
    response = requests.post(
        "https://api.gptzero.me/v2/predict/text",
        headers={
            "x-api-key": api_key,
            "Accept": "application/json",
            "Content-Type": "application/json",
            "User-Agent": "Mozilla/5.0",
        },
        json={"document": text},
        timeout=REQUEST_TIMEOUT,
    )
    if not response.ok:
        raise RuntimeError(f"GPTZero HTTP {response.status_code}: {response.text[:500]}")

    data = response.json()
    document = data.get("documents", [None])[0] if isinstance(data, dict) and data.get("documents") else data
    if not isinstance(document, dict):
        raise RuntimeError(f"GPTZero unexpected response shape: {str(data)[:500]}")

    probabilities = document.get("class_probabilities", {}) or {}
    return {
        "gptzero_predicted_class": document.get("predicted_class"),
        "gptzero_document_classification": document.get("document_classification"),
        "gptzero_result_message": document.get("result_message"),
        "gptzero_result_sub_message": document.get("result_sub_message"),
        "gptzero_confidence_category": document.get("confidence_category"),
        "gptzero_confidence_score": document.get("confidence_score"),
        "gptzero_prob_ai": probabilities.get("ai"),
        "gptzero_prob_human": probabilities.get("human"),
        "gptzero_prob_mixed": probabilities.get("mixed"),
        "gptzero_overall_burstiness": document.get("overall_burstiness"),
        "gptzero_num_paragraphs": len(document.get("paragraphs", []) or []),
        "gptzero_num_sentences": len(document.get("sentences", []) or []),
        "gptzero_raw_json": json.dumps(data, ensure_ascii=False),
    }


ensure_folder_exists(OUTPUT_CSV)
ensure_folder_exists(FAILED_CSV)

all_files, skipped_pdfs = list_target_files(AI_TRIM_ROOT)
if not all_files:
    raise FileNotFoundError(f"No .txt files found under: {AI_TRIM_ROOT}")

experiment_names = get_experiment_file_names(EXPERIMENT_FOLDER)
processed_paths, processed_names = get_already_processed_paths_and_names(OUTPUT_CSV)

print(f"Found {len(all_files)} TXT files under: {AI_TRIM_ROOT}")
print(f"Skipped {len(skipped_pdfs)} PDF files intentionally.")
print(f"Experiment TXT file names to skip: {len(experiment_names)}")
print(f"Already processed file paths in CSV: {len(processed_paths)}")
print(f"Already processed file names in CSV: {len(processed_names)}")

results = []
failed_calls = []
skipped_due_to_experiment = []
skipped_due_to_csv_path = []
skipped_due_to_csv_name = []

for index, file_path in enumerate(all_files, start=1):
    file_name = os.path.basename(file_path)

    if file_name in experiment_names:
        skipped_due_to_experiment.append(file_name)
        print(f"[{index}/{len(all_files)}] Skipping experiment file name: {file_name}")
        continue
    if file_path in processed_paths:
        skipped_due_to_csv_path.append(file_path)
        print(f"[{index}/{len(all_files)}] Skipping already processed path: {file_path}")
        continue
    if file_name in processed_names:
        skipped_due_to_csv_name.append(file_name)
        print(f"[{index}/{len(all_files)}] Skipping already processed file name from CSV: {file_name}")
        continue

    text = clean_text(extract_text_from_txt(file_path))
    text_for_apis = truncate_text(text)
    language = detect_language_simple(text_for_apis)
    row = {
        "subfolder_name": get_subfolder_name(file_path, AI_TRIM_ROOT),
        "file_name": file_name,
        "file_path": file_path,
        "extension": os.path.splitext(file_name)[1].lower(),
        "word_count": count_words(text),
        "language_guess": language,
        "extraction_method": "txt-local",
        "chars_sent_to_apis": len(text_for_apis),
    }

    print(f"[{index}/{len(all_files)}] Processing TXT: {file_name}")

    if not text_for_apis:
        for tool in ("GPTZero", "Pangram", "Sapling", "Isgen"):
            failed_calls.append({"file_path": file_path, "tool": tool, "error": "No text found in TXT file"})
        row.update({
            "gptzero_predicted_class": None,
            "pangram_prediction_short": None,
            "sapling_score": None,
            "isgen_ai_score": None,
        })
    else:
        calls = (
            ("GPTZero", lambda: call_gptzero_text(text_for_apis, GPTZERO_API_KEY)),
            ("Pangram", lambda: call_pangram(text_for_apis, PANGRAM_API_KEY)),
            ("Sapling", lambda: call_sapling(text_for_apis, SAPLING_API_KEY)),
            ("Isgen", lambda: call_isgen(text_for_apis, language, ISGEN_RAPIDAPI_KEY)),
        )
        for tool_name, call in calls:
            result = request_with_retries(tool_name, call, failed_calls, file_path)
            if result is not None:
                row.update(result)

    results.append(row)

    current_df = pd.DataFrame(results)
    if os.path.exists(OUTPUT_CSV):
        try:
            current_df = pd.concat([pd.read_csv(OUTPUT_CSV), current_df], ignore_index=True)
        except Exception:
            pass
    current_df = current_df.drop_duplicates(subset=["file_path"], keep="last")
    current_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    pd.DataFrame(failed_calls).to_csv(FAILED_CSV, index=False, encoding="utf-8-sig")

print("DONE")
print(f"Results CSV: {OUTPUT_CSV}")
print(f"Failures CSV: {FAILED_CSV}")
print(f"Total skipped PDF files: {len(skipped_pdfs)}")
print(f"Skipped experiment files: {len(skipped_due_to_experiment)}")
print(f"Skipped processed paths: {len(skipped_due_to_csv_path)}")
print(f"Skipped processed file names: {len(skipped_due_to_csv_name)}")

if failed_calls:
    print("Failed API calls:")
    for item in failed_calls:
        print(f"  Tool={item['tool']} | File={item['file_path']} | Error={item['error']}")

## Testing-sample branch

### Code block 2 — Run the three testing-sample subsets through the detectors

This block applies the detector workflow to the project’s testing samples: the trimmed AI-free sample, the humanized sample, and the AI-generated sample. The goal is to compare detector behavior across known source types.

In [ ]:
import os
import random
import re
import time

import numpy as np
import pandas as pd
import requests

PROJECT_DATA_PATH = os.environ["PROJECT_DATA_PATH"]
TESTING_SAMPLES_ROOT = PROJECT_DATA_PATH
OUTPUT_CSV = os.environ.get(
    "AI_DETECTION_OUTPUT_CSV",
    os.path.join(PROJECT_DATA_PATH, "ai_detection_results.csv"),
)

SUBFOLDERS = {
    "ai_free": "Trimmed AI-Free Sample",
    "humanized": "Humanized Sample",
    "ai_full": "AI-Generated Sample",
}

GPTZERO_API_KEY = os.environ["GPTZERO_API_KEY"]
PANGRAM_API_KEY = os.environ["PANGRAM_API_KEY"]
SAPLING_API_KEY = os.environ["SAPLING_API_KEY"]
ISGEN_API_KEY = os.environ["ISGEN_API_KEY"]

GPTZERO_URL = "https://api.gptzero.me/v2/predict/text"
PANGRAM_URL = "https://api.pangram.com/ai-content-detection"
SAPLING_URL = "https://api.sapling.ai/api/v1/aidetect"
ISGEN_URL = "https://isgen.ai/api/v1/detect"

SLEEP_BETWEEN_CALLS = 2.0
MAX_RETRIES = 3
REQUEST_TIMEOUT = 180
RESUME_IF_OUTPUT_EXISTS = True


def safe_str(value):
    return "" if pd.isna(value) else str(value)


def normalize_percent(value):
    try:
        value = float(value)
    except (TypeError, ValueError):
        return np.nan

    if value <= 1.000001:
        value *= 100
    return round(value, 2)


def word_count(text):
    return len(re.findall(r"\S+", text or ""))


def read_txt(path):
    with open(path, "r", encoding="utf-8", errors="replace") as file:
        return file.read()


def sleep_brief():
    time.sleep(SLEEP_BETWEEN_CALLS)


def request_with_retries(method, url, **kwargs):
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.request(
                method,
                url,
                timeout=REQUEST_TIMEOUT,
                **kwargs,
            )
            if response.status_code == 200:
                return response
            last_error = RuntimeError(
                f"HTTP {response.status_code}: {response.text[:1200]}"
            )
        except requests.RequestException as error:
            last_error = error

        if attempt < MAX_RETRIES:
            delay = SLEEP_BETWEEN_CALLS * attempt + random.uniform(0.2, 1.0)
            time.sleep(delay)

    raise last_error


def call_gptzero(text):
    response = request_with_retries(
        "POST",
        GPTZERO_URL,
        headers={
            "x-api-key": GPTZERO_API_KEY,
            "Content-Type": "application/json",
        },
        json={"document": text},
    )
    return response.json()


def parse_gptzero(payload):
    ai_rate = np.nan
    main_result = ""

    def parse_dict(data):
        nonlocal ai_rate, main_result
        if not isinstance(data, dict):
            return

        if not main_result:
            for key in ("predicted_class", "document_classification", "classification"):
                if data.get(key) is not None:
                    main_result = str(data[key])
                    break

        if pd.isna(ai_rate):
            for key in (
                "prob_ai",
                "completely_generated_prob",
                "average_generated_prob",
                "ai_probability",
            ):
                if data.get(key) is not None:
                    ai_rate = normalize_percent(data[key])
                    break

    if isinstance(payload, dict):
        parse_dict(payload)

        for key in ("document", "result", "prediction"):
            if isinstance(payload.get(key), dict):
                parse_dict(payload[key])

        for key in ("documents", "results", "predictions"):
            if isinstance(payload.get(key), list) and payload[key]:
                parse_dict(payload[key][0])

    return ai_rate, main_result


def call_pangram(text):
    response = request_with_retries(
        "POST",
        PANGRAM_URL,
        headers={
            "Authorization": f"Bearer {PANGRAM_API_KEY}",
            "Content-Type": "application/json",
        },
        json={"text": text},
    )
    return response.json()


def parse_pangram(payload):
    ai_rate = np.nan
    main_result = ""

    def parse_dict(data):
        nonlocal ai_rate, main_result
        if not isinstance(data, dict):
            return

        if pd.isna(ai_rate):
            for key in ("fraction_ai", "ai_probability", "score", "probability_ai"):
                if data.get(key) is not None:
                    ai_rate = normalize_percent(data[key])
                    break

        if not main_result:
            for key in ("prediction_short", "prediction", "label", "verdict"):
                if data.get(key) is not None:
                    main_result = str(data[key])
                    break

    if isinstance(payload, dict):
        parse_dict(payload)
        for key in ("result", "data"):
            parse_dict(payload.get(key))

    if not main_result and pd.notna(ai_rate):
        main_result = "Likely AI" if ai_rate >= 50 else "Likely Human"

    return ai_rate, main_result


def call_sapling(text):
    response = request_with_retries(
        "POST",
        SAPLING_URL,
        headers={"Content-Type": "application/json"},
        json={"key": SAPLING_API_KEY, "text": text},
    )
    return response.json()


def parse_sapling(payload):
    ai_rate = np.nan
    main_result = ""

    if isinstance(payload, dict):
        for key in ("score", "ai_score", "sapling_score"):
            if payload.get(key) is not None:
                ai_rate = normalize_percent(payload[key])
                break

        is_ai = payload.get("is_ai")
        if isinstance(is_ai, bool):
            main_result = "Likely AI" if is_ai else "Likely Human"

        if not main_result:
            for key in ("label", "prediction", "verdict"):
                if payload.get(key) is not None:
                    main_result = str(payload[key])
                    break

    if not main_result and pd.notna(ai_rate):
        main_result = "Likely AI" if ai_rate >= 50 else "Likely Human"

    return ai_rate, main_result


def call_isgen(text):
    response = request_with_retries(
        "POST",
        ISGEN_URL,
        headers={
            "Authorization": f"Bearer {ISGEN_API_KEY}",
            "Content-Type": "application/json",
        },
        json={"text": text},
    )
    return response.json()


def parse_isgen(payload):
    ai_rate = np.nan
    main_result = ""

    def parse_dict(data):
        nonlocal ai_rate, main_result
        if not isinstance(data, dict):
            return

        if pd.isna(ai_rate):
            for key in ("ai_score", "summary_ai", "score_ai", "prob_ai"):
                if data.get(key) is not None:
                    ai_rate = normalize_percent(data[key])
                    break

        if not main_result:
            for key in ("label", "prediction", "verdict", "classification"):
                if data.get(key) is not None:
                    main_result = str(data[key])
                    break

        if not main_result:
            scores = {
                "AI": normalize_percent(data.get("summary_ai")),
                "Human": normalize_percent(data.get("summary_human")),
                "Mixed": normalize_percent(data.get("summary_mixed")),
            }
            scores = {
                label: score for label, score in scores.items() if pd.notna(score)
            }
            if scores:
                main_result = max(scores, key=scores.get)

    if isinstance(payload, dict):
        parse_dict(payload)
        for key in ("result", "data", "document"):
            parse_dict(payload.get(key))

    if not main_result and pd.notna(ai_rate):
        main_result = "Likely AI" if ai_rate >= 50 else "Likely Human"

    return ai_rate, main_result


def run_all_tools(text):
    results = {}
    tools = (
        ("gptzero", call_gptzero, parse_gptzero),
        ("pangram", call_pangram, parse_pangram),
        ("sapling", call_sapling, parse_sapling),
        ("isgen", call_isgen, parse_isgen),
    )

    for name, caller, parser in tools:
        ai_rate, main_result = parser(caller(text))
        results[f"{name}_ai_rate_percent"] = ai_rate
        results[f"{name}_main_result"] = main_result
        sleep_brief()

    return results


VERSION_ORDER = ["ai_free", "humanized", "ai_full"]
BASE_FIELDS = [
    "word_count",
    "gptzero_ai_rate_percent",
    "gptzero_main_result",
    "pangram_ai_rate_percent",
    "pangram_main_result",
    "sapling_ai_rate_percent",
    "sapling_main_result",
    "isgen_ai_rate_percent",
    "isgen_main_result",
]
FINAL_COLUMNS = ["file_name"] + [
    f"{version}_{field}" for version in VERSION_ORDER for field in BASE_FIELDS
]


def is_version_complete(row, version):
    for field in BASE_FIELDS:
        value = row.get(f"{version}_{field}", np.nan)
        if pd.isna(value) or not safe_str(value):
            return False
    return True


folder_to_files = {}
file_name_sets = []

for version, folder_name in SUBFOLDERS.items():
    folder_path = os.path.join(TESTING_SAMPLES_ROOT, folder_name)
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Missing folder: {folder_path}")

    files = sorted(
        file_name
        for file_name in os.listdir(folder_path)
        if file_name.lower().endswith(".txt")
    )
    folder_to_files[version] = folder_path
    file_name_sets.append(set(files))

shared_file_names = sorted(set.intersection(*file_name_sets))
if not shared_file_names:
    raise ValueError("No shared .txt file names found across the required subfolders.")

print(f"Shared file count: {len(shared_file_names)}")

if RESUME_IF_OUTPUT_EXISTS and os.path.exists(OUTPUT_CSV):
    out = pd.read_csv(OUTPUT_CSV)
    if "file_name" not in out.columns:
        raise ValueError("Existing output CSV is missing the file_name column.")
else:
    out = pd.DataFrame(columns=FINAL_COLUMNS)

for column in FINAL_COLUMNS:
    if column not in out.columns:
        out[column] = np.nan if column.endswith(("_percent", "word_count")) else ""

out = out[FINAL_COLUMNS]
existing_map = {safe_str(row["file_name"]): index for index, row in out.iterrows()}
fail_log = []

for file_index, file_name in enumerate(shared_file_names, start=1):
    print(f"[{file_index}/{len(shared_file_names)}] {file_name}")

    if file_name in existing_map:
        row_index = existing_map[file_name]
    else:
        row_index = len(out)
        out.loc[row_index, "file_name"] = file_name
        existing_map[file_name] = row_index

    for version in VERSION_ORDER:
        text_path = os.path.join(folder_to_files[version], file_name)

        if not os.path.exists(text_path):
            message = f"{file_name} | {version} | missing file"
            fail_log.append(message)
            print(f"  MISSING: {version}")
            continue

        if RESUME_IF_OUTPUT_EXISTS and is_version_complete(out.loc[row_index], version):
            print(f"  SKIP COMPLETE: {version}")
            continue

        try:
            text = read_txt(text_path)
            out.at[row_index, f"{version}_word_count"] = word_count(text)
            print(
                f"  RUNNING: {version} | words={out.at[row_index, f'{version}_word_count']}"
            )

            results = run_all_tools(text)
            for field, value in results.items():
                out.at[row_index, f"{version}_{field}"] = value

        except Exception as error:
            fail_log.append(f"{file_name} | {version} | {error}")
            print(f"  FAILED: {version} -> {error}")

        # Persist each completed version so interrupted runs can resume.
        out = out[FINAL_COLUMNS].sort_values("file_name").reset_index(drop=True)
        existing_map = {
            safe_str(row["file_name"]): index for index, row in out.iterrows()
        }
        row_index = existing_map[file_name]
        out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

out = out[FINAL_COLUMNS].sort_values("file_name").reset_index(drop=True)
out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"Output CSV: {OUTPUT_CSV}")
print(f"Shape: {out.shape}")
print(f"Failure count: {len(fail_log)}")
for failure in fail_log:
    print(f"- {failure}")

# Testing Sample

### Code block 3 — Revised robust testing-sample run

This block is a refined version of the testing-sample detector run. It uses the API approach that matched the earlier successful script and is kept here for reproducibility and comparison with the earlier attempt.

In [ ]:
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# Configure local data and output locations outside the notebook.
PROJECT_DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", "data"))
TESTING_SAMPLES_ROOT = Path(
    os.environ.get("TESTING_SAMPLES_ROOT", PROJECT_DATA_DIR / "testing_samples")
)
OUTPUT_CSV = Path(
    os.environ.get("OUTPUT_CSV", PROJECT_DATA_DIR / "detection_results.csv")
)
BACKUP_JSONL = Path(
    os.environ.get("BACKUP_JSONL", PROJECT_DATA_DIR / "detection_log.jsonl")
)
BACKUP_TXT = Path(os.environ.get("BACKUP_TXT", PROJECT_DATA_DIR / "detection_log.txt"))

SUBFOLDERS = {
    "ai_free": "Trimmed AI-Free Sample",
    "humanized": "Humanized Sample",
    "ai_full": "AI-Generated Sample",
}

GPTZERO_API_KEY = os.environ.get("GPTZERO_API_KEY", "")
PANGRAM_API_KEY = os.environ.get("PANGRAM_API_KEY", "")
SAPLING_API_KEY = os.environ.get("SAPLING_API_KEY", "")
ISGEN_RAPIDAPI_KEY = os.environ.get("ISGEN_RAPIDAPI_KEY", "")

SLEEP_BETWEEN_CALLS = 2.0
MAX_RETRIES = 2
WAIT_SECONDS_BETWEEN_RETRIES = 12
REQUEST_TIMEOUT = 180
RESUME_IF_OUTPUT_EXISTS = True
MAX_TEXT_CHARS = 50000


def normalize_percent(value):
    try:
        value = float(value)
        if value <= 1.000001:
            value *= 100
        return round(value, 2)
    except (TypeError, ValueError):
        return np.nan


def count_words(text):
    return len(re.findall(r"\S+", text or ""))


def read_txt(path):
    for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            with open(path, "r", encoding=encoding, errors="strict") as file:
                return file.read()
        except (OSError, UnicodeDecodeError):
            continue

    with open(path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()


def clean_text(text):
    if not text:
        return ""
    text = text.replace("\x00", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def truncate_text(text, max_chars=MAX_TEXT_CHARS):
    """Prefer paragraph or sentence boundaries when applying API length limits."""
    if not text or len(text) <= max_chars:
        return text or ""

    cut = text[:max_chars]
    last_paragraph = cut.rfind("\n\n")
    if last_paragraph > max_chars * 0.6:
        return cut[:last_paragraph].strip()

    last_punctuation = max(
        cut.rfind(". "),
        cut.rfind("! "),
        cut.rfind("? "),
        cut.rfind("۔ "),
        cut.rfind("\n"),
    )
    if last_punctuation > max_chars * 0.6:
        return cut[: last_punctuation + 1].strip()
    return cut.strip()


def detect_language_simple(text):
    if not text:
        return "en"
    arabic_chars = re.findall(r"[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF]", text)
    return "ar" if len(arabic_chars) / len(text) > 0.03 else "en"


def request_with_retries(method, url, **kwargs):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.request(method, url, timeout=REQUEST_TIMEOUT, **kwargs)
            if 200 <= response.status_code < 300:
                return response
            last_error = RuntimeError(
                f"HTTP {response.status_code}: {response.text[:1200]}"
            )
        except requests.RequestException as error:
            last_error = error

        if attempt < MAX_RETRIES:
            time.sleep(WAIT_SECONDS_BETWEEN_RETRIES)

    raise last_error


def save_progress(df, final_columns):
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    df = df[final_columns].sort_values("file_name").reset_index(drop=True)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    return df


def append_jsonl(record):
    BACKUP_JSONL.parent.mkdir(parents=True, exist_ok=True)
    with open(BACKUP_JSONL, "a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")


def append_txt(line):
    BACKUP_TXT.parent.mkdir(parents=True, exist_ok=True)
    with open(BACKUP_TXT, "a", encoding="utf-8") as file:
        file.write(line + "\n")


def log_tool_result(
    file_name,
    version_key,
    tool_name,
    word_count_value,
    chars_sent,
    ai_rate,
    main_result,
    status,
    error_msg=None,
    raw_payload=None,
):
    record = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "file_name": file_name,
        "version": version_key,
        "tool": tool_name,
        "word_count": word_count_value,
        "chars_sent_to_api": chars_sent,
        "status": status,
        "ai_rate_percent": None if pd.isna(ai_rate) else ai_rate,
        "main_result": main_result,
        "error": error_msg,
        "raw_payload": raw_payload,
    }
    append_jsonl(record)
    append_txt(
        f"[{record['timestamp']}] file={file_name} | version={version_key} | "
        f"tool={tool_name} | status={status} | word_count={word_count_value} | "
        f"chars_sent={chars_sent} | ai_rate_percent={record['ai_rate_percent']} | "
        f"main_result={main_result} | error={error_msg or ''}"
    )


def print_row_snapshot(df, row_idx, prefix):
    columns = [
        "file_name",
        f"{prefix}_word_count",
        f"{prefix}_gptzero_ai_rate_percent",
        f"{prefix}_gptzero_main_result",
        f"{prefix}_pangram_ai_rate_percent",
        f"{prefix}_pangram_main_result",
        f"{prefix}_sapling_ai_rate_percent",
        f"{prefix}_sapling_main_result",
        f"{prefix}_isgen_ai_rate_percent",
        f"{prefix}_isgen_main_result",
    ]
    columns = [column for column in columns if column in df.columns]
    print("\nCURRENT SAVED SNAPSHOT")
    print(df.loc[[row_idx], columns].to_string(index=False))
    print()


def call_gptzero(text, api_key):
    response = request_with_retries(
        "POST",
        "https://api.gptzero.me/v2/predict/text",
        headers={
            "x-api-key": api_key,
            "Accept": "application/json",
            "Content-Type": "application/json",
            "User-Agent": "Mozilla/5.0",
        },
        json={"document": text},
    )
    data = response.json()
    document = (
        data.get("documents", [None])[0]
        if isinstance(data, dict) and data.get("documents")
        else data
    )

    if not isinstance(document, dict):
        raise RuntimeError(f"GPTZero unexpected response shape: {str(data)[:500]}")

    probabilities = document.get("class_probabilities", {}) or {}
    return {
        "raw": data,
        "gptzero_predicted_class": document.get("predicted_class"),
        "gptzero_document_classification": document.get("document_classification"),
        "gptzero_result_message": document.get("result_message"),
        "gptzero_result_sub_message": document.get("result_sub_message"),
        "gptzero_confidence_category": document.get("confidence_category"),
        "gptzero_confidence_score": document.get("confidence_score"),
        "gptzero_prob_ai": probabilities.get("ai"),
        "gptzero_prob_human": probabilities.get("human"),
        "gptzero_prob_mixed": probabilities.get("mixed"),
        "gptzero_overall_burstiness": document.get("overall_burstiness"),
        "gptzero_num_paragraphs": len(document.get("paragraphs", []) or []),
        "gptzero_num_sentences": len(document.get("sentences", []) or []),
    }


def call_pangram(text, api_key):
    response = request_with_retries(
        "POST",
        "https://text.api.pangram.com/v3",
        headers={"Content-Type": "application/json", "x-api-key": api_key},
        json={"text": text, "public_dashboard_link": False},
    )
    data = response.json()
    return {
        "raw": data,
        "pangram_prediction_short": data.get("prediction_short"),
        "pangram_prediction": data.get("prediction"),
        "pangram_headline": data.get("headline"),
        "pangram_fraction_ai": data.get("fraction_ai"),
        "pangram_fraction_ai_assisted": data.get("fraction_ai_assisted"),
        "pangram_fraction_human": data.get("fraction_human"),
        "pangram_num_ai_segments": data.get("num_ai_segments"),
        "pangram_num_ai_assisted_segments": data.get("num_ai_assisted_segments"),
        "pangram_num_human_segments": data.get("num_human_segments"),
        "pangram_num_windows": len(data.get("windows", []) or []),
    }


def call_sapling(text, api_key):
    response = request_with_retries(
        "POST",
        "https://api.sapling.ai/api/v1/aidetect",
        json={"key": api_key, "text": text},
    )
    data = response.json()
    return {
        "raw": data,
        "sapling_score": data.get("score"),
        "sapling_text_len_chars": len(data.get("text", "") or ""),
        "sapling_num_sentence_scores": len(data.get("sentence_scores", []) or []),
    }


def call_isgen(text, lang, api_key):
    response = request_with_retries(
        "POST",
        "https://ai-detection4.p.rapidapi.com/v1/ai-detection-rapid-api",
        headers={
            "x-rapidapi-key": api_key,
            "x-rapidapi-host": "ai-detection4.p.rapidapi.com",
            "Content-Type": "application/json",
        },
        json={"text": text, "lang": lang},
    )
    data = response.json()
    summary = data.get("summary", {}) or {}
    return {
        "raw": data,
        "isgen_ai_score": data.get("ai_score"),
        "isgen_lang": data.get("lang"),
        "isgen_total_num_words": data.get("total_num_words"),
        "isgen_summary_ai": summary.get("ai"),
        "isgen_summary_human": summary.get("human"),
        "isgen_summary_mixed": summary.get("mixed"),
        "isgen_num_paragraphs": len(data.get("paragraphs", []) or []),
    }


def normalize_gptzero_result(result):
    ai_rate = normalize_percent(result.get("gptzero_prob_ai"))
    main_result = (
        result.get("gptzero_predicted_class")
        or result.get("gptzero_document_classification")
        or ""
    )
    if not main_result and pd.notna(ai_rate):
        main_result = "Likely AI" if ai_rate >= 50 else "Likely Human"
    return ai_rate, main_result


def normalize_pangram_result(result):
    ai_rate = normalize_percent(result.get("pangram_fraction_ai"))
    main_result = (
        result.get("pangram_prediction_short") or result.get("pangram_prediction") or ""
    )
    if not main_result and pd.notna(ai_rate):
        main_result = "Likely AI" if ai_rate >= 50 else "Likely Human"
    return ai_rate, main_result


def normalize_sapling_result(result):
    ai_rate = normalize_percent(result.get("sapling_score"))
    main_result = (
        "" if pd.isna(ai_rate) else "Likely AI" if ai_rate >= 50 else "Likely Human"
    )
    return ai_rate, main_result


def normalize_isgen_result(result):
    ai_rate = normalize_percent(result.get("isgen_ai_score"))
    values = {
        label: normalize_percent(result.get(key))
        for label, key in {
            "AI": "isgen_summary_ai",
            "Human": "isgen_summary_human",
            "Mixed": "isgen_summary_mixed",
        }.items()
        if result.get(key) is not None
    }
    values = {label: value for label, value in values.items() if pd.notna(value)}

    if values:
        return ai_rate, max(values, key=values.get)
    if pd.notna(ai_rate):
        return ai_rate, "Likely AI" if ai_rate >= 50 else "Likely Human"
    return ai_rate, ""


VERSION_ORDER = ["ai_free", "humanized", "ai_full"]
BASE_FIELDS = [
    "word_count",
    "gptzero_ai_rate_percent",
    "gptzero_main_result",
    "pangram_ai_rate_percent",
    "pangram_main_result",
    "sapling_ai_rate_percent",
    "sapling_main_result",
    "isgen_ai_rate_percent",
    "isgen_main_result",
]
FINAL_COLUMNS = ["file_name"] + [
    f"{version}_{field}" for version in VERSION_ORDER for field in BASE_FIELDS
]

TOOLS = [
    (
        "gptzero",
        "GPTZero",
        lambda text, lang: call_gptzero(text, GPTZERO_API_KEY),
        normalize_gptzero_result,
    ),
    (
        "pangram",
        "Pangram",
        lambda text, lang: call_pangram(text, PANGRAM_API_KEY),
        normalize_pangram_result,
    ),
    (
        "sapling",
        "Sapling",
        lambda text, lang: call_sapling(text, SAPLING_API_KEY),
        normalize_sapling_result,
    ),
    (
        "isgen",
        "Isgen",
        lambda text, lang: call_isgen(text, lang, ISGEN_RAPIDAPI_KEY),
        normalize_isgen_result,
    ),
]

folder_to_files = {}
file_sets = []
for version_key, folder_name in SUBFOLDERS.items():
    folder_path = TESTING_SAMPLES_ROOT / folder_name
    if not folder_path.is_dir():
        raise FileNotFoundError(f"Missing folder: {folder_path}")

    files = sorted(
        path.name for path in folder_path.iterdir() if path.suffix.lower() == ".txt"
    )
    folder_to_files[version_key] = {"folder_path": folder_path, "files": files}
    file_sets.append(set(files))

shared_file_names = sorted(set.intersection(*file_sets))
if not shared_file_names:
    raise ValueError("No shared .txt files found in the configured folders.")

print(f"Shared file count: {len(shared_file_names)}")
print(f"Output CSV: {OUTPUT_CSV}")
print(f"JSONL log: {BACKUP_JSONL}")
print(f"Text log: {BACKUP_TXT}")

if RESUME_IF_OUTPUT_EXISTS and OUTPUT_CSV.exists():
    out = pd.read_csv(OUTPUT_CSV)
    if "file_name" not in out.columns:
        out = pd.DataFrame(columns=FINAL_COLUMNS)
else:
    out = pd.DataFrame(columns=FINAL_COLUMNS)

for column in FINAL_COLUMNS:
    if column not in out.columns:
        out[column] = (
            np.nan if column.endswith(("_word_count", "_ai_rate_percent")) else ""
        )
out = out[FINAL_COLUMNS]


def get_or_create_row_idx(df, file_name):
    matches = df.index[df["file_name"].astype(str) == str(file_name)].tolist()
    if matches:
        return matches[0]
    row_idx = len(df)
    df.loc[row_idx, "file_name"] = file_name
    return row_idx


failure_log = []
for file_i, file_name in enumerate(shared_file_names, start=1):
    print("=" * 110)
    print(f"[{file_i}/{len(shared_file_names)}] {file_name}")
    row_idx = get_or_create_row_idx(out, file_name)

    for version_key in VERSION_ORDER:
        folder_path = folder_to_files[version_key]["folder_path"]
        text_path = folder_path / file_name

        if not text_path.exists():
            message = f"{file_name} | {version_key} | missing file"
            failure_log.append(message)
            print("  MISSING:", message)
            append_txt(message)
            continue

        cleaned_text = clean_text(read_txt(text_path))
        text_for_api = truncate_text(cleaned_text)
        language = detect_language_simple(text_for_api)
        word_count = count_words(cleaned_text)

        out.at[row_idx, f"{version_key}_word_count"] = word_count
        out = save_progress(out, FINAL_COLUMNS)
        print(
            f"  RUNNING: {version_key} | words={word_count} | "
            f"lang={language} | chars_sent={len(text_for_api)}"
        )

        if not text_for_api.strip():
            print(
                "    Empty text after cleaning/truncation. Marking all tools as ERROR."
            )
            for tool_key, _, _, _ in TOOLS:
                out.at[row_idx, f"{version_key}_{tool_key}_ai_rate_percent"] = np.nan
                out.at[row_idx, f"{version_key}_{tool_key}_main_result"] = "ERROR"
            out = save_progress(out, FINAL_COLUMNS)
            continue

        for tool_key, tool_name, call_tool, normalize_result in TOOLS:
            try:
                result = call_tool(text_for_api, language)
                ai_rate, main_result = normalize_result(result)
                out.at[row_idx, f"{version_key}_{tool_key}_ai_rate_percent"] = ai_rate
                out.at[row_idx, f"{version_key}_{tool_key}_main_result"] = main_result
                print(
                    f"    {tool_name} OK -> ai_rate_percent={ai_rate} | main_result={main_result}"
                )
                log_tool_result(
                    file_name,
                    version_key,
                    tool_name,
                    word_count,
                    len(text_for_api),
                    ai_rate,
                    main_result,
                    "OK",
                    raw_payload=result.get("raw"),
                )
            except Exception as error:
                out.at[row_idx, f"{version_key}_{tool_key}_ai_rate_percent"] = np.nan
                out.at[row_idx, f"{version_key}_{tool_key}_main_result"] = "ERROR"
                failure_log.append(
                    f"{file_name} | {version_key} | {tool_name} | {error}"
                )
                print(f"    {tool_name} FAILED -> {error}")
                log_tool_result(
                    file_name,
                    version_key,
                    tool_name,
                    word_count,
                    len(text_for_api),
                    np.nan,
                    "ERROR",
                    "FAILED",
                    error_msg=str(error),
                )

            out = save_progress(out, FINAL_COLUMNS)
            time.sleep(SLEEP_BETWEEN_CALLS)

        print_row_snapshot(out, row_idx, version_key)

out = save_progress(out, FINAL_COLUMNS)
print("\n" + "=" * 110)
print("DONE")
print("=" * 110)
print("Output CSV:", OUTPUT_CSV)
print("Backup JSONL:", BACKUP_JSONL)
print("Backup TXT:", BACKUP_TXT)
print("Shape:", out.shape)
print("\nFailure count:", len(failure_log))
if failure_log:
    print("\nFirst 100 failures:")
    for failure in failure_log[:100]:
        print("-", failure)

### Code block 4 — Retry failed Pangram calls for the testing sample

This block reruns only the Pangram failures caused by insufficient credits or similar issues and writes the repaired values back into the same testing-sample result set.

In [ ]:
import os
import re
import time

import numpy as np
import pandas as pd
import requests

TESTING_SAMPLES_ROOT = os.environ["PANGRAM_SAMPLES_ROOT"]
OUTPUT_CSV = os.environ["PANGRAM_OUTPUT_CSV"]
BACKUP_TXT = os.environ["PANGRAM_BACKUP_TXT"]
PANGRAM_API_KEY = os.environ["PANGRAM_API_KEY"]

SUBFOLDERS = {
    "ai_free": "Trimmed AI-Free Sample",
    "humanized": "Humanized Sample",
    "ai_full": "AI-Generated Sample",
}

MAX_TEXT_CHARS = 50_000
MAX_RETRIES = 2
WAIT_SECONDS_BETWEEN_RETRIES = 12
REQUEST_TIMEOUT = 180

FAILED_PANGRAM_CASES = [
    ("mhmdrm_yhb_wd_ai_trimmed.txt", "humanized"),
    ("mhmdrm_yhb_wd_ai_trimmed.txt", "ai_full"),
    ("paper_shredder_design_report.docx_ai_trimmed.txt", "ai_free"),
    ("paper_shredder_design_report.docx_ai_trimmed.txt", "humanized"),
    ("paper_shredder_design_report.docx_ai_trimmed.txt", "ai_full"),
    ("smart-aquarium-control-kit-final-report_ai_trimmed.txt", "ai_free"),
    ("smart-aquarium-control-kit-final-report_ai_trimmed.txt", "humanized"),
    ("smart-aquarium-control-kit-final-report_ai_trimmed.txt", "ai_full"),
    ("solar_panels_cleaning_system_report_ai_trimmed.txt", "ai_free"),
    ("solar_panels_cleaning_system_report_ai_trimmed.txt", "humanized"),
    ("solar_panels_cleaning_system_report_ai_trimmed.txt", "ai_full"),
    ("wave-power-generator-final-report_ai_trimmed.txt", "ai_free"),
    ("wave-power-generator-final-report_ai_trimmed.txt", "humanized"),
    ("wave-power-generator-final-report_ai_trimmed.txt", "ai_full"),
]


def normalize_percent(value):
    try:
        value = float(value)
        if value <= 1.000001:
            value *= 100
        return round(value, 2)
    except (TypeError, ValueError):
        return np.nan


def read_txt(path):
    for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            with open(path, "r", encoding=encoding, errors="strict") as file:
                return file.read()
        except (OSError, UnicodeDecodeError):
            continue

    with open(path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()


def clean_text(text):
    if not text:
        return ""
    text = text.replace("\x00", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def truncate_text(text, max_chars=MAX_TEXT_CHARS):
    if not text or len(text) <= max_chars:
        return text or ""

    cut = text[:max_chars]
    last_paragraph = cut.rfind("\n\n")
    if last_paragraph > max_chars * 0.6:
        return cut[:last_paragraph].strip()

    # Prefer a natural boundary when truncation is necessary.
    last_boundary = max(
        cut.rfind(". "),
        cut.rfind("! "),
        cut.rfind("? "),
        cut.rfind("۔ "),
        cut.rfind("\n"),
    )
    if last_boundary > max_chars * 0.6:
        return cut[: last_boundary + 1].strip()

    return cut.strip()


def request_with_retries(method, url, **kwargs):
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.request(
                method,
                url,
                timeout=REQUEST_TIMEOUT,
                **kwargs,
            )
            if 200 <= response.status_code < 300:
                return response
            last_error = RuntimeError(
                f"HTTP {response.status_code}: {response.text[:1200]}"
            )
        except requests.RequestException as error:
            last_error = error

        if attempt < MAX_RETRIES:
            time.sleep(WAIT_SECONDS_BETWEEN_RETRIES)

    raise last_error


def append_txt(line, path):
    with open(path, "a", encoding="utf-8") as file:
        file.write(f"{line}\n")


def log_txt_result(
    file_name,
    version,
    status,
    ai_rate=None,
    main_result=None,
    error_msg=None,
    chars_sent=None,
):
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    line = (
        f"[{timestamp}] file={file_name} | version={version} | "
        f"tool=Pangram-Retry | status={status} | "
        f"chars_sent={chars_sent if chars_sent is not None else ''} | "
        f"ai_rate_percent={'' if pd.isna(ai_rate) else ai_rate} | "
        f"main_result={main_result if main_result is not None else ''} | "
        f"error={error_msg if error_msg else ''}"
    )
    append_txt(line, BACKUP_TXT)


def call_pangram(text, api_key):
    response = request_with_retries(
        "POST",
        "https://text.api.pangram.com/v3",
        headers={
            "Content-Type": "application/json",
            "x-api-key": api_key,
        },
        json={
            "text": text,
            "public_dashboard_link": False,
        },
    )
    data = response.json()

    return {
        "raw": data,
        "pangram_prediction_short": data.get("prediction_short"),
        "pangram_prediction": data.get("prediction"),
        "pangram_headline": data.get("headline"),
        "pangram_fraction_ai": data.get("fraction_ai"),
        "pangram_fraction_ai_assisted": data.get("fraction_ai_assisted"),
        "pangram_fraction_human": data.get("fraction_human"),
        "pangram_num_ai_segments": data.get("num_ai_segments"),
        "pangram_num_ai_assisted_segments": data.get("num_ai_assisted_segments"),
        "pangram_num_human_segments": data.get("num_human_segments"),
        "pangram_num_windows": len(data.get("windows", []) or []),
    }


def normalize_pangram_result(result):
    ai_rate = normalize_percent(result.get("pangram_fraction_ai"))
    main_result = (
        result.get("pangram_prediction_short") or result.get("pangram_prediction") or ""
    )
    if not main_result and pd.notna(ai_rate):
        main_result = "Likely AI" if ai_rate >= 50 else "Likely Human"
    return ai_rate, main_result


if not os.path.exists(OUTPUT_CSV):
    raise FileNotFoundError(f"CSV not found: {OUTPUT_CSV}")

df = pd.read_csv(OUTPUT_CSV)

if "file_name" not in df.columns:
    raise ValueError("CSV must contain a file_name column.")

for version in SUBFOLDERS:
    rate_column = f"{version}_pangram_ai_rate_percent"
    result_column = f"{version}_pangram_main_result"
    if rate_column not in df.columns:
        df[rate_column] = np.nan
    if result_column not in df.columns:
        df[result_column] = ""

successes = []
failures = []

print(f"Using CSV backup: {OUTPUT_CSV}")
print(f"Appending TXT backup to: {BACKUP_TXT}")

for index, (file_name, version) in enumerate(FAILED_PANGRAM_CASES, start=1):
    chars_sent = None
    print("=" * 100)
    print(f"[{index}/{len(FAILED_PANGRAM_CASES)}] Retrying Pangram")
    print(f"  file_name = {file_name}")
    print(f"  version   = {version}")

    matches = df.index[df["file_name"].astype(str) == str(file_name)].tolist()
    if not matches:
        message = f"Row not found in CSV for file_name={file_name}"
        print("  FAILED:", message)
        failures.append((file_name, version, message))
        log_txt_result(
            file_name,
            version,
            "FAILED",
            ai_rate=np.nan,
            main_result="ERROR",
            error_msg=message,
        )
        continue

    row_idx = matches[0]
    txt_path = os.path.join(TESTING_SAMPLES_ROOT, SUBFOLDERS[version], file_name)

    if not os.path.exists(txt_path):
        message = f"TXT file not found: {txt_path}"
        print("  FAILED:", message)
        failures.append((file_name, version, message))
        log_txt_result(
            file_name,
            version,
            "FAILED",
            ai_rate=np.nan,
            main_result="ERROR",
            error_msg=message,
        )
        continue

    try:
        text_for_api = truncate_text(clean_text(read_txt(txt_path)))
        chars_sent = len(text_for_api)

        if not text_for_api.strip():
            raise RuntimeError("Empty text after cleaning/truncation")

        result = call_pangram(text_for_api, PANGRAM_API_KEY)
        ai_rate, main_result = normalize_pangram_result(result)

        df.at[row_idx, f"{version}_pangram_ai_rate_percent"] = ai_rate
        df.at[row_idx, f"{version}_pangram_main_result"] = main_result
        df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

        log_txt_result(
            file_name,
            version,
            "OK",
            ai_rate=ai_rate,
            main_result=main_result,
            chars_sent=chars_sent,
        )
        print(f"  SUCCESS -> ai_rate_percent={ai_rate} | main_result={main_result}")
        successes.append((file_name, version, ai_rate, main_result))

    except Exception as error:
        message = str(error)
        print(f"  FAILED -> {message}")
        failures.append((file_name, version, message))
        log_txt_result(
            file_name,
            version,
            "FAILED",
            ai_rate=np.nan,
            main_result="ERROR",
            error_msg=message,
            chars_sent=chars_sent,
        )

print("\n" + "=" * 110)
print("DONE")
print("=" * 110)
print(f"Updated CSV: {OUTPUT_CSV}")
print(f"Updated TXT backup: {BACKUP_TXT}")
print(f"Success count: {len(successes)}")
print(f"Failure count: {len(failures)}")

if successes:
    print("\nSUCCESSFUL UPDATES:")
    for file_name, version, ai_rate, main_result in successes:
        print(
            f"- {file_name} | {version} | "
            f"ai_rate_percent={ai_rate} | main_result={main_result}"
        )

if failures:
    print("\nFAILED UPDATES:")
    for file_name, version, message in failures:
        print(f"- {file_name} | {version} | {message}")

print("\nUPDATED ROW SNAPSHOTS:")
seen = set()
for file_name, _ in FAILED_PANGRAM_CASES:
    if file_name in seen:
        continue
    seen.add(file_name)

    row = df[df["file_name"].astype(str) == str(file_name)]
    if not row.empty:
        columns = [
            "file_name",
            "ai_free_pangram_ai_rate_percent",
            "ai_free_pangram_main_result",
            "humanized_pangram_ai_rate_percent",
            "humanized_pangram_main_result",
            "ai_full_pangram_ai_rate_percent",
            "ai_full_pangram_main_result",
        ]
        columns = [column for column in columns if column in df.columns]
        print(row[columns].to_string(index=False))
        print()

## Main written-assignment detection run (revised copy)

### Code block 5 — Revised main-corpus detector run

This block is a later version of the main **AI Trim** detector script. It is kept in the notebook because it documents the refined procedure used after earlier failures or adjustments.

### Code block 6 — Retry failed Pangram calls for the main corpus

This block repairs Pangram failures in the main written-assignment results without rerunning the entire detector stack.

In [ ]:
!pip -q install requests pandas

import json
import os
import re
import time

import pandas as pd
import requests
from google.colab import drive

drive.mount("/content/drive")

PANGRAM_API_KEY = os.environ.get("PANGRAM_API_KEY")
OUTPUT_CSV = os.environ.get("OUTPUT_CSV")
PANGRAM_RETRY_FAILURES_CSV = os.environ.get(
    "PANGRAM_RETRY_FAILURES_CSV",
    os.path.join(os.path.dirname(OUTPUT_CSV or ""), "pangram_retry_failures.csv"),
)
FAILED_PANGRAM_FILES = json.loads(os.environ.get("FAILED_PANGRAM_FILES_JSON", "[]"))

MAX_TEXT_CHARS = 50000
MAX_RETRIES = 2
WAIT_SECONDS_BETWEEN_RETRIES = 12
REQUEST_TIMEOUT = 180

PANGRAM_COLUMNS = [
    "pangram_prediction_short",
    "pangram_prediction",
    "pangram_headline",
    "pangram_fraction_ai",
    "pangram_fraction_ai_assisted",
    "pangram_fraction_human",
    "pangram_num_ai_segments",
    "pangram_num_ai_assisted_segments",
    "pangram_num_human_segments",
    "pangram_num_windows",
    "pangram_raw_json",
]


def ensure_folder_exists(path):
    folder = os.path.dirname(path)
    if folder:
        os.makedirs(folder, exist_ok=True)


def count_words(text):
    return len(re.findall(r"\S+", text)) if text else 0


def detect_language_simple(text):
    if not text:
        return "en"
    arabic_chars = re.findall(r"[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF]", text)
    return "ar" if len(arabic_chars) / max(len(text), 1) > 0.03 else "en"


def clean_text(text):
    if not text:
        return ""
    text = text.replace("\x00", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def truncate_text(text, max_chars=MAX_TEXT_CHARS):
    if not text or len(text) <= max_chars:
        return text or ""

    cut = text[:max_chars]
    # Prefer a paragraph or sentence boundary to avoid sending a partial passage.
    last_para = cut.rfind("\n\n")
    if last_para > max_chars * 0.6:
        return cut[:last_para].strip()

    last_punct = max(
        cut.rfind(". "),
        cut.rfind("! "),
        cut.rfind("? "),
        cut.rfind("۔ "),
        cut.rfind("\n"),
    )
    if last_punct > max_chars * 0.6:
        return cut[: last_punct + 1].strip()
    return cut.strip()


def extract_text_from_txt(file_path):
    for encoding in ["utf-8", "utf-8-sig", "cp1252", "latin-1"]:
        try:
            with open(file_path, "r", encoding=encoding, errors="strict") as file:
                return file.read()
        except Exception:
            pass

    with open(file_path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()


def call_pangram(text, api_key):
    response = requests.post(
        "https://text.api.pangram.com/v3",
        headers={"Content-Type": "application/json", "x-api-key": api_key},
        json={"text": text, "public_dashboard_link": False},
        timeout=REQUEST_TIMEOUT,
    )
    if not 200 <= response.status_code < 300:
        raise RuntimeError(f"Pangram HTTP {response.status_code}: {response.text[:500]}")

    data = response.json()
    return {
        "pangram_prediction_short": data.get("prediction_short"),
        "pangram_prediction": data.get("prediction"),
        "pangram_headline": data.get("headline"),
        "pangram_fraction_ai": data.get("fraction_ai"),
        "pangram_fraction_ai_assisted": data.get("fraction_ai_assisted"),
        "pangram_fraction_human": data.get("fraction_human"),
        "pangram_num_ai_segments": data.get("num_ai_segments"),
        "pangram_num_ai_assisted_segments": data.get("num_ai_assisted_segments"),
        "pangram_num_human_segments": data.get("num_human_segments"),
        "pangram_num_windows": len(data.get("windows", []) or []),
        "pangram_raw_json": json.dumps(data, ensure_ascii=False),
    }


def request_with_retries(func):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return func()
        except Exception as error:
            last_error = str(error)
            print(f"    Attempt {attempt}/{MAX_RETRIES} failed -> {error}")
            if "Insufficient credits" in last_error:
                break
            if attempt < MAX_RETRIES:
                time.sleep(WAIT_SECONDS_BETWEEN_RETRIES)
    raise RuntimeError(last_error or "Unknown error")


if not PANGRAM_API_KEY:
    raise ValueError("Set PANGRAM_API_KEY in the environment.")
if not OUTPUT_CSV:
    raise ValueError("Set OUTPUT_CSV in the environment.")
if not isinstance(FAILED_PANGRAM_FILES, list):
    raise ValueError("FAILED_PANGRAM_FILES_JSON must be a JSON array of TXT file paths.")

ensure_folder_exists(OUTPUT_CSV)
ensure_folder_exists(PANGRAM_RETRY_FAILURES_CSV)

if not os.path.exists(OUTPUT_CSV):
    raise FileNotFoundError(f"Main CSV not found: {OUTPUT_CSV}")

df = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")

if "file_path" not in df.columns:
    raise ValueError("The main CSV does not contain a 'file_path' column.")

for column in PANGRAM_COLUMNS:
    if column not in df.columns:
        df[column] = None

retry_failures = []

print(f"Loaded main CSV with {len(df)} rows.")
print(f"Will retry Pangram for {len(FAILED_PANGRAM_FILES)} files.")

for index, file_path in enumerate(FAILED_PANGRAM_FILES, start=1):
    print("=" * 100)
    print(f"[{index}/{len(FAILED_PANGRAM_FILES)}] Retrying Pangram: {file_path}")

    if not os.path.exists(file_path):
        error = "TXT file not found"
        print(f"  FAILED -> {error}")
        retry_failures.append({"file_path": file_path, "tool": "Pangram", "error": error})
        continue

    try:
        text = clean_text(extract_text_from_txt(file_path))
        word_count = count_words(text)
        language = detect_language_simple(text)
        text_for_api = truncate_text(text)

        print(f"  Word count: {word_count}")
        print(f"  Language guess: {language}")
        print(f"  Text chars sent to Pangram: {len(text_for_api)}")

        if not text_for_api.strip():
            raise RuntimeError("No text found in TXT file")

        pangram_result = request_with_retries(
            lambda: call_pangram(text_for_api, PANGRAM_API_KEY)
        )

        print(
            f"  Pangram done -> prediction_short={pangram_result.get('pangram_prediction_short')}, "
            f"fraction_ai={pangram_result.get('pangram_fraction_ai')}"
        )

        mask = df["file_path"].astype(str) == str(file_path)
        if mask.any():
            for key, value in pangram_result.items():
                df.loc[mask, key] = value
            print("  Updated existing row(s) in main CSV for this file.")
        else:
            new_row = {column: None for column in df.columns}
            new_row["file_path"] = file_path
            if "file_name" in new_row:
                new_row["file_name"] = os.path.basename(file_path)
            if "word_count" in new_row:
                new_row["word_count"] = word_count
            if "language_guess" in new_row:
                new_row["language_guess"] = language
            if "chars_sent_to_apis" in new_row:
                new_row["chars_sent_to_apis"] = len(text_for_api)
            new_row.update(pangram_result)
            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
            print("  File path was not already in CSV, so a new row was appended.")

        df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
        print(f"  Saved updated main CSV -> {OUTPUT_CSV}")

    except Exception as error:
        error_message = str(error)
        print(f"  FAILED -> {error_message}")
        retry_failures.append(
            {"file_path": file_path, "tool": "Pangram", "error": error_message}
        )
        df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
        pd.DataFrame(retry_failures).to_csv(
            PANGRAM_RETRY_FAILURES_CSV, index=False, encoding="utf-8-sig"
        )

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
pd.DataFrame(retry_failures).to_csv(
    PANGRAM_RETRY_FAILURES_CSV, index=False, encoding="utf-8-sig"
)

print("\n" + "=" * 120)
print("DONE")
print(f"Updated main CSV: {OUTPUT_CSV}")
print(f"Pangram retry failures CSV: {PANGRAM_RETRY_FAILURES_CSV}")
print(f"Remaining Pangram retry failures: {len(retry_failures)}")

if retry_failures:
    print("\nFAILED RETRIES:")
    for item in retry_failures:
        print(f"  Tool={item['tool']} | File={item['file_path']} | Error={item['error']}")
else:
    print("\nAll Pangram retries succeeded.")

## Recovery and reconstruction utilities

### Code block 7 — Reconstruct a detector CSV from TXT log files

This block rebuilds a structured CSV when only printed TXT logs from earlier runs are available. It is a recovery utility used to preserve results after accidental overwrites or interrupted runs.

In [ ]:
import os
import re

import pandas as pd

# Configure paths through environment variables to avoid embedding private locations.
RESULTS1_TXT = os.environ.get("RESULTS1_TXT")
RESULTS2_TXT = os.environ.get("RESULTS2_TXT")
OUTPUT_CSV = os.environ.get("OUTPUT_CSV")

if not all((RESULTS1_TXT, RESULTS2_TXT, OUTPUT_CSV)):
    raise ValueError(
        "Set RESULTS1_TXT, RESULTS2_TXT, and OUTPUT_CSV environment variables."
    )


def ensure_folder_exists(path):
    folder = os.path.dirname(path)
    if folder:
        os.makedirs(folder, exist_ok=True)


def parse_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


# Pangram and Sapling log scores use a 0--1 scale; the output table uses percent.
def pct_from_unit_interval(value):
    if value is None:
        return None
    return round(value * 100.0, 4)


def infer_binary_label_from_percent(value):
    if value is None or pd.isna(value):
        return ""
    return "AI" if float(value) >= 50 else "Human"


def normalize_gptzero_result(predicted_class, document_classification):
    predicted = (predicted_class or "").strip().lower()
    classification = (document_classification or "").strip().upper()

    if predicted == "ai" or classification == "AI_ONLY":
        return "AI"
    if predicted == "human" or classification == "HUMAN_ONLY":
        return "Human"
    if predicted == "mixed" or classification == "MIXED":
        return "Mixed"
    return predicted_class or document_classification or ""


def normalize_pangram_result(prediction_short):
    prediction = (prediction_short or "").strip().lower()
    labels = {"ai": "AI", "human": "Human", "mixed": "Mixed"}
    return labels.get(prediction, prediction_short or "")


def extract_root_from_path(path):
    return os.path.basename(os.path.dirname(path)) if path else ""


def empty_row(file_path, file_name, word_count):
    return {
        "root": extract_root_from_path(file_path),
        "file_name": file_name,
        "file_path": file_path,
        "word_count": word_count,
        # GPTZero logs include a class but not a probability value.
        "gptzero_ai_rate_percent": None,
        "gptzero_main_result": "",
        "pangram_ai_rate_percent": None,
        "pangram_main_result": "",
        "sapling_ai_rate_percent": None,
        "sapling_main_result": "",
        "isgen_ai_rate_percent": None,
        "isgen_main_result": "",
    }


def parse_results1(text):
    rows = {}
    pattern = re.compile(
        r"=+\s*"
        r"\[\d+/\d+\]\s+Processing TXT:\s+(?P<file_name>.+?)\n"
        r"\s*Path:\s+(?P<file_path>.+?)\n"
        r"\s*Subfolder:\s+(?P<subfolder>.+?)\n"
        r"\s*Word count:\s+(?P<word_count>\d+)\n"
        r"\s*Language guess:\s+(?P<lang>.+?)\n"
        r"\s*Text chars sent to APIs:\s+(?P<chars>\d+)\n"
        r"(?P<body>.*?)(?=\n=+\s*\[\d+/\d+\]\s+Processing TXT:|\Z)",
        re.DOTALL,
    )

    for match in pattern.finditer(text):
        file_name = match.group("file_name").strip()
        file_path = match.group("file_path").strip()
        body = match.group("body")
        row = empty_row(file_path, file_name, int(match.group("word_count")))

        gptzero = re.search(
            r"GPTZero done -> predicted_class=(.*?), document_classification=(.*?)\n",
            body,
        )
        if gptzero:
            row["gptzero_main_result"] = normalize_gptzero_result(
                gptzero.group(1).strip(), gptzero.group(2).strip()
            )

        pangram = re.search(
            r"Pangram done -> prediction_short=(.*?), fraction_ai=([0-9eE+\-.]+)",
            body,
        )
        if pangram:
            row["pangram_ai_rate_percent"] = pct_from_unit_interval(
                parse_float(pangram.group(2))
            )
            row["pangram_main_result"] = normalize_pangram_result(
                pangram.group(1).strip()
            )

        sapling = re.search(r"Sapling done -> score=([0-9eE+\-.]+)", body)
        if sapling:
            row["sapling_ai_rate_percent"] = pct_from_unit_interval(
                parse_float(sapling.group(1))
            )
            row["sapling_main_result"] = infer_binary_label_from_percent(
                row["sapling_ai_rate_percent"]
            )

        isgen = re.search(r"Isgen done -> ai_score=([0-9eE+\-.]+), lang=", body)
        if isgen:
            score = parse_float(isgen.group(1))
            row["isgen_ai_rate_percent"] = (
                round(score, 4) if score is not None else None
            )
            row["isgen_main_result"] = infer_binary_label_from_percent(
                row["isgen_ai_rate_percent"]
            )

        rows[file_path] = row

    return rows


def parse_results2_and_update(text, rows):
    pattern = re.compile(
        r"=+\s*"
        r"\[\d+/\d+\]\s+Retrying Pangram:\s+(?P<file_path>.+?)\n"
        r"\s*Word count:\s+(?P<word_count>\d+)\n"
        r"\s*Language guess:\s+(?P<lang>.+?)\n"
        r"\s*Text chars sent to Pangram:\s+(?P<chars>\d+)\n"
        r"(?P<body>.*?)(?=\n=+\s*\[\d+/\d+\]\s+Retrying Pangram:|\Z)",
        re.DOTALL,
    )

    for match in pattern.finditer(text):
        file_path = match.group("file_path").strip()
        if file_path not in rows:
            rows[file_path] = empty_row(
                file_path,
                os.path.basename(file_path),
                int(match.group("word_count")),
            )

        pangram = re.search(
            r"Pangram done -> prediction_short=(.*?), fraction_ai=([0-9eE+\-.]+)",
            match.group("body"),
        )
        if pangram:
            rows[file_path]["pangram_ai_rate_percent"] = pct_from_unit_interval(
                parse_float(pangram.group(2))
            )
            rows[file_path]["pangram_main_result"] = normalize_pangram_result(
                pangram.group(1).strip()
            )

    return rows


for path in (RESULTS1_TXT, RESULTS2_TXT):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Could not find: {path}")

with open(RESULTS1_TXT, encoding="utf-8", errors="ignore") as file:
    rows = parse_results1(file.read())

with open(RESULTS2_TXT, encoding="utf-8", errors="ignore") as file:
    parse_results2_and_update(file.read(), rows)

final_columns = [
    "root",
    "file_name",
    "word_count",
    "gptzero_ai_rate_percent",
    "gptzero_main_result",
    "pangram_ai_rate_percent",
    "pangram_main_result",
    "sapling_ai_rate_percent",
    "sapling_main_result",
    "isgen_ai_rate_percent",
    "isgen_main_result",
]

df = pd.DataFrame(rows.values())
for column in final_columns:
    if column not in df:
        df[column] = None

df = df[final_columns].copy()


def natural_file_key(name):
    match = re.match(r"^\s*(\d+)", str(name))
    return (int(match.group(1)) if match else 10**9, str(name))


df = df.sort_values(
    by="file_name", key=lambda series: series.map(natural_file_key)
).reset_index(drop=True)

ensure_folder_exists(OUTPUT_CSV)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"Saved reconstructed CSV to: {OUTPUT_CSV}")
print(df.head())

## KFUPM Second-Year branch

### Code block 10 — Run AI detection for KFUPM Second Year Reports and Coding

This block keeps the **reports** workflow as a four-detector written-text run, but updates the **coding** workflow to match the dedicated **AI Code Detection** notebook.

For the coding folder it now uses the code-specific Pangram pipeline:
- one TXT file is treated as one submission
- internal source files are parsed from path headers and separators
- Pangram is called per internal code file or chunk
- a single submission-level score is aggregated at the end

In [ ]:
# Install dependencies when running this cell in a fresh notebook environment.
!pip -q install requests pandas numpy

import json
import os
import re
import time
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import requests

PROJECT_DATA_PATH = Path(os.environ["PROJECT_DATA_PATH"]).expanduser()
REPORTS_DIR = PROJECT_DATA_PATH / "Reports"
CODING_DIR = PROJECT_DATA_PATH / "Coding"
OUTPUT_DIR = PROJECT_DATA_PATH / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPORTS_OUTPUT_CSV = OUTPUT_DIR / "reports_ai_detection_results.csv"
REPORTS_BACKUP_JSONL = OUTPUT_DIR / "reports_ai_detection_results_backup.jsonl"
REPORTS_BACKUP_TXT = OUTPUT_DIR / "reports_ai_detection_results_backup.txt"
CODING_OUTPUT_CSV = OUTPUT_DIR / "coding_ai_detection_results.csv"
CODING_BACKUP_JSONL = OUTPUT_DIR / "coding_ai_detection_results_backup.jsonl"
CODING_BACKUP_TXT = OUTPUT_DIR / "coding_ai_detection_results_backup.txt"

GPTZERO_API_KEY = os.environ["GPTZERO_API_KEY"]
PANGRAM_API_KEY = os.environ["PANGRAM_API_KEY"]
SAPLING_API_KEY = os.environ["SAPLING_API_KEY"]
ISGEN_RAPIDAPI_KEY = os.environ["ISGEN_RAPIDAPI_KEY"]

SLEEP_BETWEEN_CALLS = 2.0
MAX_RETRIES = 2
WAIT_SECONDS_BETWEEN_RETRIES = 12
REQUEST_TIMEOUT = 180
RESUME_IF_OUTPUT_EXISTS = True
MAX_TEXT_CHARS = 50_000
MAX_WORDS_PER_CHUNK = 2_500
SEPARATOR_MARKERS = {"############", "##########", "==========", "-----", "__________"}
KNOWN_CODE_EXTS = {
    ".js", ".jsx", ".ts", ".tsx", ".json", ".css", ".scss", ".sass", ".less",
    ".html", ".htm", ".vue", ".md", ".xml", ".yml", ".yaml", ".graphql", ".gql",
    ".sql", ".mjs", ".cjs", ".py", ".java", ".cpp", ".c", ".h", ".hpp", ".php",
}


def normalize_percent(value):
    try:
        value = float(value)
        if value <= 1.000001:
            value *= 100
        return round(value, 2)
    except (TypeError, ValueError):
        return np.nan


def count_words(text):
    return len(re.findall(r"\S+", text)) if text else 0


def read_txt(path):
    for encoding, errors in (("utf-8", None), ("utf-8-sig", None), ("cp1252", "ignore")):
        try:
            with open(path, "r", encoding=encoding, errors=errors) as file:
                return file.read()
        except (OSError, UnicodeDecodeError):
            continue
    raise OSError(f"Could not read file: {path}")


def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace("\x00", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def truncate_text(text, max_chars=MAX_TEXT_CHARS):
    return text[:max_chars] if len(text) > max_chars else text


def detect_language_simple(text):
    return "ar" if re.search(r"[\u0600-\u06FF]", text or "") else "en"


def label_from_score_50(score_percent):
    if pd.isna(score_percent):
        return ""
    return "AI" if float(score_percent) >= 50 else "Human"


def append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")


def append_txt(path, line):
    with open(path, "a", encoding="utf-8") as file:
        file.write(line + "\n")


def call_gptzero(text):
    url = "https://api.gptzero.me/v2/predict/text"
    headers = {"x-api-key": GPTZERO_API_KEY, "Content-Type": "application/json"}
    payload = {"document": text}
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            data = response.json()
            documents = data.get("documents")
            document = documents[0] if isinstance(documents, list) and documents else {}
            score = normalize_percent(document.get("completely_generated_prob"))
            return score, label_from_score_50(score), data
        except Exception as error:
            last_error = error
            if attempt < MAX_RETRIES:
                time.sleep(WAIT_SECONDS_BETWEEN_RETRIES)

    return np.nan, "", {"error": str(last_error)}


def call_pangram_text(text, language_hint=None):
    url = "https://text.api.pangram.com/v3"
    headers = {"API-Key": PANGRAM_API_KEY, "Content-Type": "application/json"}
    payload = {"text": text}
    if language_hint:
        payload["language"] = language_hint

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            data = response.json()
            score = normalize_percent(data.get("score", data.get("ai_probability")))
            return score, label_from_score_50(score), data
        except Exception as error:
            last_error = error
            if attempt < MAX_RETRIES:
                time.sleep(WAIT_SECONDS_BETWEEN_RETRIES)

    return np.nan, "", {"error": str(last_error)}


def call_sapling(text):
    url = "https://api.sapling.ai/api/v1/aidetect"
    headers = {"Content-Type": "application/json"}
    payload = {"key": SAPLING_API_KEY, "text": text}
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            data = response.json()
            score = normalize_percent(data.get("score"))
            return score, label_from_score_50(score), data
        except Exception as error:
            last_error = error
            if attempt < MAX_RETRIES:
                time.sleep(WAIT_SECONDS_BETWEEN_RETRIES)

    return np.nan, "", {"error": str(last_error)}


def call_isgen(text):
    url = "https://ai-detection4.p.rapidapi.com/v1/ai-detection-rapid-api"
    headers = {
        "x-rapidapi-key": ISGEN_RAPIDAPI_KEY,
        "x-rapidapi-host": "ai-detection4.p.rapidapi.com",
        "Content-Type": "application/json",
    }
    payload = {"text": text}
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            data = response.json()
            score = normalize_percent(data.get("ai_score"))
            return score, label_from_score_50(score), data
        except Exception as error:
            last_error = error
            if attempt < MAX_RETRIES:
                time.sleep(WAIT_SECONDS_BETWEEN_RETRIES)

    return np.nan, "", {"error": str(last_error)}


def normalize_text_code(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\x00", "")
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def sanitize_rel_path(rel_path):
    parts = []
    for part in PurePosixPath(rel_path.replace("\\", "/").strip()).parts:
        if part in ("", ".", ".."):
            continue
        part = re.sub(r'[<>:"|?*]', "_", part).replace("\x00", "").strip()
        if part:
            parts.append(part)
    cleaned = "/".join(parts)
    return cleaned[:240] if cleaned else "main.js"


def looks_like_path_line(line):
    line = line.strip()
    if not line or line in SEPARATOR_MARKERS or line.startswith(("http://", "https://")) or len(line) > 260:
        return False

    if not any(line.lower().endswith(ext) for ext in KNOWN_CODE_EXTS):
        return False

    code_tokens = (
        "import ", "from ", "const ", "let ", "var ", "function ", "class ",
        "<", ">", "{", "}", ";", "=>", "(", ")", "=", "require(",
    )
    if any(token in line for token in code_tokens):
        return False

    return bool(re.fullmatch(r"[A-Za-z0-9_\-./\\ @()\[\]{}+]+", line))


def parse_multifile_txt_submission(raw_text):
    text = normalize_text_code(raw_text)
    files = []
    current_path = None
    current_lines = []

    def flush_current():
        nonlocal current_path, current_lines
        body = "\n".join(current_lines).strip()
        if body:
            files.append({
                "path": sanitize_rel_path(current_path or f"file_{len(files) + 1}.txt"),
                "content": body,
            })
        current_path = None
        current_lines = []

    # Path-like lines and separator markers delimit source files within one submission export.
    for line in text.split("\n"):
        stripped = line.strip()
        if stripped in SEPARATOR_MARKERS:
            flush_current()
        elif looks_like_path_line(stripped):
            flush_current()
            current_path = stripped
        else:
            current_lines.append(line)

    flush_current()
    return files or ([{"path": "main.txt", "content": text}] if text else [])


def split_text_by_words(text, max_words=MAX_WORDS_PER_CHUNK):
    tokens = re.findall(r"\S+|\s+", text, flags=re.UNICODE)
    chunks = []
    current = []
    word_count = 0

    for token in tokens:
        if token.isspace():
            current.append(token)
            continue
        if word_count + 1 > max_words and current:
            chunks.append("".join(current).strip())
            current = [token]
            word_count = 1
        else:
            current.append(token)
            word_count += 1

    if current:
        chunks.append("".join(current).strip())
    return [chunk for chunk in chunks if chunk]


def call_pangram_code_chunk(code_text):
    url = "https://text.api.pangram.com/v3"
    headers = {"API-Key": PANGRAM_API_KEY, "Content-Type": "application/json"}
    payload = {"text": code_text}
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            data = response.json()
            return normalize_percent(data.get("score", data.get("ai_probability"))), data
        except Exception as error:
            last_error = error
            if attempt < MAX_RETRIES:
                time.sleep(WAIT_SECONDS_BETWEEN_RETRIES)

    return np.nan, {"error": str(last_error)}


def detect_submission_with_pangram(parsed_files):
    file_level = []
    raw_payloads = []
    weighted_sum = 0.0
    total_words = 0

    for source_file in parsed_files:
        code = source_file["content"].strip()
        word_count = count_words(code)
        if word_count == 0:
            continue

        chunk_scores = []
        chunk_payloads = []
        for chunk in split_text_by_words(code):
            chunk_words = count_words(chunk)
            score, payload = call_pangram_code_chunk(chunk)
            chunk_scores.append((score, chunk_words))
            chunk_payloads.append({"chunk_words": chunk_words, "response": payload})
            time.sleep(SLEEP_BETWEEN_CALLS)

        valid_scores = [(score, words) for score, words in chunk_scores if not pd.isna(score)]
        file_score = (
            sum(score * words for score, words in valid_scores) / sum(words for _, words in valid_scores)
            if valid_scores else np.nan
        )

        file_level.append({
            "path": source_file["path"],
            "word_count": word_count,
            "score_percent": round(file_score, 2) if not pd.isna(file_score) else np.nan,
        })
        raw_payloads.append({
            "path": source_file["path"],
            "word_count": word_count,
            "chunks": chunk_payloads,
        })

        # Aggregate file scores by source length so larger files contribute proportionally.
        if not pd.isna(file_score):
            weighted_sum += file_score * word_count
            total_words += word_count

    submission_score = weighted_sum / total_words if total_words else np.nan
    return {
        "submission_score_percent": round(submission_score, 2) if not pd.isna(submission_score) else np.nan,
        "submission_result": label_from_score_50(submission_score),
        "submission_word_count": total_words,
        "file_level": file_level,
        "raw_payloads": raw_payloads,
    }


def load_existing_records(csv_path, key_col="file_name"):
    if RESUME_IF_OUTPUT_EXISTS and csv_path.exists():
        try:
            records = pd.read_csv(csv_path)
            if key_col in records.columns:
                return set(records[key_col].astype(str))
        except Exception as error:
            print(f"Warning: could not load {csv_path}: {error}")
    return set()


def append_result_to_csv(csv_path, row):
    existing = pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame()
    combined = pd.concat([existing, pd.DataFrame([row])], ignore_index=True)
    combined.drop_duplicates(subset=["file_name"], keep="last", inplace=True)
    combined.to_csv(csv_path, index=False, encoding="utf-8-sig")


if not REPORTS_DIR.exists():
    raise FileNotFoundError(f"Reports folder not found: {REPORTS_DIR}")
if not CODING_DIR.exists():
    raise FileNotFoundError(f"Coding folder not found: {CODING_DIR}")

report_files = sorted(path for path in REPORTS_DIR.rglob("*.txt") if path.is_file())
existing_report_names = load_existing_records(REPORTS_OUTPUT_CSV)

print(f"Reports files found: {len(report_files)}")
for index, path in enumerate(report_files, start=1):
    file_name = path.name
    if file_name in existing_report_names:
        print(f"[Reports {index}/{len(report_files)}] Skipping existing: {file_name}")
        continue

    print(f"[Reports {index}/{len(report_files)}] {file_name}")
    text = truncate_text(clean_text(read_txt(path)))
    word_count = count_words(text)

    gptzero_score, gptzero_result, gptzero_raw = call_gptzero(text)
    time.sleep(SLEEP_BETWEEN_CALLS)
    pangram_score, pangram_result, pangram_raw = call_pangram_text(text, detect_language_simple(text))
    time.sleep(SLEEP_BETWEEN_CALLS)
    sapling_score, sapling_result, sapling_raw = call_sapling(text)
    time.sleep(SLEEP_BETWEEN_CALLS)
    isgen_score, isgen_result, isgen_raw = call_isgen(text)

    row = {
        "file_name": file_name,
        "word_count": word_count,
        "gptzero_ai_rate_percent": gptzero_score,
        "gptzero_result": gptzero_result,
        "pangram_ai_rate_percent": pangram_score,
        "pangram_result": pangram_result,
        "sapling_ai_rate_percent": sapling_score,
        "sapling_result": sapling_result,
        "isgen_ai_rate_percent": isgen_score,
        "isgen_result": isgen_result,
    }
    append_jsonl(REPORTS_BACKUP_JSONL, {
        "file_name": file_name,
        "word_count": word_count,
        "gptzero_raw": gptzero_raw,
        "pangram_raw": pangram_raw,
        "sapling_raw": sapling_raw,
        "isgen_raw": isgen_raw,
    })
    append_txt(REPORTS_BACKUP_TXT, json.dumps(row, ensure_ascii=False))
    append_result_to_csv(REPORTS_OUTPUT_CSV, row)

coding_files = sorted(path for path in CODING_DIR.rglob("*.txt") if path.is_file())
existing_coding_names = load_existing_records(CODING_OUTPUT_CSV)

print(f"Coding files found: {len(coding_files)}")
for index, path in enumerate(coding_files, start=1):
    file_name = path.name
    if file_name in existing_coding_names:
        print(f"[Coding {index}/{len(coding_files)}] Skipping existing: {file_name}")
        continue

    print(f"[Coding {index}/{len(coding_files)}] {file_name}")
    detection = detect_submission_with_pangram(parse_multifile_txt_submission(read_txt(path)))
    row = {
        "file_name": file_name,
        "word_count": detection["submission_word_count"],
        "pangram_ai_rate_percent": detection["submission_score_percent"],
        "pangram_result": detection["submission_result"],
    }
    append_jsonl(CODING_BACKUP_JSONL, {
        "file_name": file_name,
        "word_count": detection["submission_word_count"],
        "file_level": detection["file_level"],
        "raw_payloads": detection["raw_payloads"],
    })
    append_txt(CODING_BACKUP_TXT, json.dumps(row, ensure_ascii=False))
    append_result_to_csv(CODING_OUTPUT_CSV, row)

print("Done")
print(f"Reports CSV: {REPORTS_OUTPUT_CSV}")
print(f"Coding CSV: {CODING_OUTPUT_CSV}")

In [ ]:
import json
import os
import re
import time
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import requests

# Configure these variables for the private data location and Pangram credentials.
PROJECT_DATA_PATH = os.environ.get("PROJECT_DATA_PATH")
REPORTS_INPUT_CSV_PATH = os.environ.get("REPORTS_INPUT_CSV_PATH")
PANGRAM_API_KEY = os.environ.get("PANGRAM_API_KEY")

if not PROJECT_DATA_PATH:
    raise EnvironmentError(
        "Set PROJECT_DATA_PATH to the directory containing Reports and Coding."
    )
if not REPORTS_INPUT_CSV_PATH:
    raise EnvironmentError("Set REPORTS_INPUT_CSV_PATH to the existing reports CSV.")
if not PANGRAM_API_KEY:
    raise EnvironmentError("Set PANGRAM_API_KEY before running detection.")

BASE_DIR = Path(PROJECT_DATA_PATH)
REPORTS_DIR = BASE_DIR / "Reports"
CODING_DIR = BASE_DIR / "Coding"
REPORTS_INPUT_CSV = Path(REPORTS_INPUT_CSV_PATH)
OUTPUT_DIR = BASE_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPORTS_OUTPUT_CSV = (
    OUTPUT_DIR / "KFUPM_Second_Year_Reports_AI_Detection_Results_PANGRAM_UPDATED.csv"
)
REPORTS_BACKUP_JSONL = (
    OUTPUT_DIR
    / "KFUPM_Second_Year_Reports_AI_Detection_Results_PANGRAM_UPDATED_BACKUP.jsonl"
)
REPORTS_BACKUP_TXT = (
    OUTPUT_DIR
    / "KFUPM_Second_Year_Reports_AI_Detection_Results_PANGRAM_UPDATED_BACKUP.txt"
)

CODING_OUTPUT_CSV = (
    OUTPUT_DIR / "KFUPM_Second_Year_Coding_AI_Detection_Results_PANGRAM_ONLY.csv"
)
CODING_BACKUP_JSONL = (
    OUTPUT_DIR
    / "KFUPM_Second_Year_Coding_AI_Detection_Results_PANGRAM_ONLY_BACKUP.jsonl"
)
CODING_BACKUP_TXT = (
    OUTPUT_DIR / "KFUPM_Second_Year_Coding_AI_Detection_Results_PANGRAM_ONLY_BACKUP.txt"
)

SLEEP_BETWEEN_CALLS = 2.0
MAX_RETRIES = 2
WAIT_SECONDS_BETWEEN_RETRIES = 12
REQUEST_TIMEOUT = 180
MAX_TEXT_CHARS = 50_000
MAX_WORDS_PER_CHUNK = 2_500
SECONDS_BETWEEN_REQUESTS = SLEEP_BETWEEN_CALLS

SEPARATOR_MARKERS = {"############", "##########", "==========", "-----", "__________"}
KNOWN_CODE_EXTS = {
    ".js",
    ".jsx",
    ".ts",
    ".tsx",
    ".json",
    ".css",
    ".scss",
    ".sass",
    ".less",
    ".html",
    ".htm",
    ".vue",
    ".md",
    ".xml",
    ".yml",
    ".yaml",
    ".graphql",
    ".gql",
    ".sql",
    ".mjs",
    ".cjs",
    ".py",
    ".java",
    ".cpp",
    ".c",
    ".h",
    ".hpp",
    ".php",
    ".cs",
    ".rb",
    ".go",
    ".rs",
    ".kt",
    ".swift",
    ".sh",
    ".ipynb",
}


def normalize_percent(value):
    try:
        value = float(value)
        if value <= 1.000001:
            value *= 100
        return round(value, 2)
    except (TypeError, ValueError):
        return np.nan


def count_words(text):
    return len(re.findall(r"\S+", text or ""))


def read_txt(path):
    for encoding, errors in (
        ("utf-8", None),
        ("utf-8-sig", None),
        ("cp1252", "ignore"),
    ):
        try:
            with open(path, "r", encoding=encoding, errors=errors) as file:
                return file.read()
        except UnicodeDecodeError:
            continue


def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace("\x00", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def truncate_text(text, max_chars=MAX_TEXT_CHARS):
    return (text or "")[:max_chars]


def detect_language_simple(text):
    return "ar" if re.search(r"[\u0600-\u06FF]", text or "") else "en"


def label_from_score_50(score_percent):
    if pd.isna(score_percent):
        return ""
    return "AI" if float(score_percent) >= 50 else "Human"


def append_jsonl(path, obj):
    with open(path, "a", encoding="utf-8") as file:
        file.write(json.dumps(obj, ensure_ascii=False) + "\n")


def append_txt(path, line):
    with open(path, "a", encoding="utf-8") as file:
        file.write(line + "\n")


def request_with_retries(method, url, **kwargs):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.request(method, url, timeout=REQUEST_TIMEOUT, **kwargs)
            if response.status_code in (429, 500, 502, 503, 504):
                print(
                    f"    Attempt {attempt}: HTTP {response.status_code}, retrying in {WAIT_SECONDS_BETWEEN_RETRIES}s"
                )
                time.sleep(WAIT_SECONDS_BETWEEN_RETRIES)
                continue
            if response.status_code >= 400:
                raise RuntimeError(
                    f"HTTP {response.status_code}: {response.text[:1000]}"
                )
            return response
        except Exception as error:
            last_error = error
            if attempt < MAX_RETRIES:
                print(f"    Attempt {attempt} failed: {error}")
                time.sleep(WAIT_SECONDS_BETWEEN_RETRIES)
            else:
                raise RuntimeError(
                    f"All retries failed. Last error: {last_error}"
                ) from error


def extract_pangram_score(data):
    score = data.get("score")
    if score is None:
        score = data.get("ai_probability")
    if score is None:
        score = data.get("fraction_ai")
    return normalize_percent(score)


def call_pangram_text(text, language_hint=None):
    payload = {"text": text}
    if language_hint:
        payload["language"] = language_hint

    response = request_with_retries(
        "POST",
        "https://text.api.pangram.com/v3",
        headers={
            "Content-Type": "application/json",
            "x-api-key": PANGRAM_API_KEY.strip(),
        },
        json=payload,
    )
    data = response.json()
    score = extract_pangram_score(data)
    return score, label_from_score_50(score), data


def sanitize_rel_path(rel_path):
    rel_path = rel_path.replace("\\", "/").strip()
    parts = []
    for part in PurePosixPath(rel_path).parts:
        if part in ("", ".", ".."):
            continue
        cleaned = re.sub(r'[<>:"|?*]', "_", part).replace("\x00", "").strip()
        if cleaned:
            parts.append(cleaned)
    cleaned_path = "/".join(parts)
    return cleaned_path[:240] if cleaned_path else "main.js"


def normalize_text_code(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\x00", "")
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def looks_like_path_line(line):
    line = line.strip()
    if (
        not line
        or line in SEPARATOR_MARKERS
        or line.startswith(("http://", "https://"))
        or len(line) > 260
    ):
        return False

    lower = line.lower()
    if not any(lower.endswith(extension) for extension in KNOWN_CODE_EXTS):
        return False

    code_tokens = [
        "import ",
        "from ",
        "const ",
        "let ",
        "var ",
        "function ",
        "class ",
        "<",
        ">",
        "{",
        "}",
        ";",
        "=>",
        "(",
        ")",
        "=",
        "require(",
    ]
    if any(token in line for token in code_tokens):
        return False

    return bool(re.fullmatch(r"[A-Za-z0-9_\-./\\ @()\[\]{}+]+", line))


def parse_multifile_txt_submission(raw_text):
    lines = normalize_text_code(raw_text).split("\n")
    files = []
    current_path = None
    current_lines = []

    def flush_current():
        nonlocal current_path, current_lines
        body = "\n".join(current_lines).strip()
        if body:
            files.append(
                {
                    "path": sanitize_rel_path(
                        current_path or f"file_{len(files) + 1}.txt"
                    ),
                    "content": body,
                }
            )
        current_path = None
        current_lines = []

    for line in lines:
        stripped = line.strip()
        if stripped in SEPARATOR_MARKERS:
            flush_current()
        elif looks_like_path_line(stripped):
            flush_current()
            current_path = stripped
        else:
            current_lines.append(line)

    flush_current()

    text = normalize_text_code(raw_text)
    return files or ([{"path": "main.txt", "content": text}] if text else [])


def split_text_by_words(text, max_words=MAX_WORDS_PER_CHUNK):
    tokens = re.findall(r"\S+|\s+", text, flags=re.UNICODE)
    chunks = []
    current = []
    current_word_count = 0

    for token in tokens:
        if token.isspace():
            current.append(token)
        elif current_word_count + 1 > max_words and current:
            chunks.append("".join(current).strip())
            current = [token]
            current_word_count = 1
        else:
            current.append(token)
            current_word_count += 1

    if current:
        chunks.append("".join(current).strip())
    return [chunk for chunk in chunks if chunk]


def call_pangram_code_chunk(code_text):
    response = request_with_retries(
        "POST",
        "https://text.api.pangram.com/v3",
        headers={
            "Content-Type": "application/json",
            "x-api-key": PANGRAM_API_KEY.strip(),
        },
        json={"text": code_text},
    )
    data = response.json()
    return extract_pangram_score(data), data


def detect_submission_with_pangram(parsed_files):
    file_level = []
    raw_payloads = []
    weighted_sum = 0.0
    total_words = 0

    for parsed_file in parsed_files:
        code = parsed_file["content"].strip()
        word_count = count_words(code)
        if word_count == 0:
            continue

        chunk_scores = []
        chunk_payloads = []
        for chunk in split_text_by_words(code):
            chunk_word_count = count_words(chunk)
            if chunk_word_count == 0:
                continue
            score, payload = call_pangram_code_chunk(chunk)
            chunk_scores.append((score, chunk_word_count))
            chunk_payloads.append(
                {"chunk_words": chunk_word_count, "response": payload}
            )
            time.sleep(SECONDS_BETWEEN_REQUESTS)

        valid_scores = [
            (score, weight) for score, weight in chunk_scores if not pd.isna(score)
        ]
        file_score = (
            sum(score * weight for score, weight in valid_scores)
            / sum(weight for _, weight in valid_scores)
            if valid_scores
            else np.nan
        )

        file_level.append(
            {
                "path": parsed_file["path"],
                "word_count": word_count,
                "score_percent": (
                    round(file_score, 2) if not pd.isna(file_score) else np.nan
                ),
            }
        )
        raw_payloads.append(
            {
                "path": parsed_file["path"],
                "word_count": word_count,
                "chunks": chunk_payloads,
            }
        )

        if not pd.isna(file_score):
            weighted_sum += file_score * word_count
            total_words += word_count

    submission_score = weighted_sum / total_words if total_words else np.nan
    return {
        "submission_score_percent": (
            round(submission_score, 2) if not pd.isna(submission_score) else np.nan
        ),
        "submission_result": label_from_score_50(submission_score),
        "submission_word_count": total_words,
        "file_level": file_level,
        "raw_payloads": raw_payloads,
    }


if not REPORTS_DIR.exists():
    raise FileNotFoundError(f"Reports folder not found: {REPORTS_DIR}")
if not CODING_DIR.exists():
    raise FileNotFoundError(f"Coding folder not found: {CODING_DIR}")
if not REPORTS_INPUT_CSV.exists():
    raise FileNotFoundError(f"Input reports CSV not found: {REPORTS_INPUT_CSV}")

reports_df = pd.read_csv(REPORTS_INPUT_CSV, encoding="utf-8-sig")
for column, default in (("pangram_ai_rate_percent", np.nan), ("pangram_result", "")):
    if column not in reports_df.columns:
        reports_df[column] = default

report_files = sorted(path for path in REPORTS_DIR.rglob("*.txt") if path.is_file())
print(f"REPORTS FILES FOUND: {len(report_files)}")

for index, path in enumerate(report_files, start=1):
    print(f"[REPORTS {index}/{len(report_files)}] {path.name}")

    text = truncate_text(clean_text(read_txt(path)))
    word_count = count_words(text)
    score, result, raw_response = call_pangram_text(text, detect_language_simple(text))

    matches = reports_df.index[
        reports_df["file_name"].astype(str) == path.name
    ].tolist()
    row_index = matches[0] if matches else len(reports_df)
    reports_df.loc[row_index, "file_name"] = path.name
    if "word_count" in reports_df.columns:
        reports_df.loc[row_index, "word_count"] = word_count
    reports_df.loc[row_index, "pangram_ai_rate_percent"] = score
    reports_df.loc[row_index, "pangram_result"] = result

    append_jsonl(
        REPORTS_BACKUP_JSONL,
        {
            "file_name": path.name,
            "word_count": word_count,
            "pangram_raw": raw_response,
        },
    )
    append_txt(
        REPORTS_BACKUP_TXT,
        json.dumps(
            {
                "file_name": path.name,
                "word_count": word_count,
                "pangram_ai_rate_percent": score,
                "pangram_result": result,
            },
            ensure_ascii=False,
        ),
    )

    reports_df.to_csv(REPORTS_OUTPUT_CSV, index=False, encoding="utf-8-sig")
    time.sleep(SLEEP_BETWEEN_CALLS)

coding_files = sorted(path for path in CODING_DIR.rglob("*.txt") if path.is_file())
print(f"CODING FILES FOUND: {len(coding_files)}")

coding_rows = []
for index, path in enumerate(coding_files, start=1):
    print(f"[CODING {index}/{len(coding_files)}] {path.name}")

    detection = detect_submission_with_pangram(
        parse_multifile_txt_submission(read_txt(path))
    )
    row = {
        "file_name": path.name,
        "word_count": detection["submission_word_count"],
        "pangram_ai_rate_percent": detection["submission_score_percent"],
        "pangram_result": detection["submission_result"],
    }
    coding_rows.append(row)

    append_jsonl(
        CODING_BACKUP_JSONL,
        {
            "file_name": path.name,
            "word_count": detection["submission_word_count"],
            "file_level": detection["file_level"],
            "raw_payloads": detection["raw_payloads"],
        },
    )
    append_txt(CODING_BACKUP_TXT, json.dumps(row, ensure_ascii=False))
    pd.DataFrame(coding_rows).to_csv(
        CODING_OUTPUT_CSV, index=False, encoding="utf-8-sig"
    )

print("DONE")
print(f"Updated Reports CSV: {REPORTS_OUTPUT_CSV}")
print(f"Reports TXT backup: {REPORTS_BACKUP_TXT}")
print(f"New Coding CSV: {CODING_OUTPUT_CSV}")
print(f"Coding TXT backup: {CODING_BACKUP_TXT}")

### Code block 8 — Combine the English, Arabic, and code detection results

This block merges result files from multiple branches of the project into one consolidated dataset for later statistical analysis and interpretation.

In [ ]:
import importlib
import json
import os
import re
import subprocess
import sys


def ensure_package(package, import_name=None):
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


for package, import_name in [
    ("pandas", "pandas"),
    ("openpyxl", "openpyxl"),
    ("xlrd", "xlrd"),
    ("gspread", "gspread"),
    ("google-auth", "google.auth"),
]:
    ensure_package(package, import_name)

import pandas as pd

DATA_DIR = os.environ.get("PROJECT_DATA_PATH")
if not DATA_DIR:
    raise EnvironmentError(
        "Set PROJECT_DATA_PATH to the directory containing the input files."
    )

MOUNT_GOOGLE_DRIVE = os.environ.get("MOUNT_GOOGLE_DRIVE", "true").lower() in {
    "1",
    "true",
    "yes",
}
GOOGLE_DRIVE_MOUNT_POINT = os.environ.get("GOOGLE_DRIVE_MOUNT_POINT", "/content/drive")

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount(GOOGLE_DRIVE_MOUNT_POINT, force_remount=True)

ENGLISH_BASENAME = "AI_Detection_Results_English"
ARABIC_BASENAME = "AI_Detection_Results_Arabic"
CODE_BASENAME = "KFUPM_code_AI_detection_results"
OUTPUT_CSV = os.path.join(DATA_DIR, "AI_Detection_Results_Combined.csv")


def find_file_by_basename(folder, basename):
    extensions = [".csv", ".xlsx", ".xls", ".gsheet"]
    candidates = [
        os.path.join(folder, basename + extension) for extension in extensions
    ]
    existing = [path for path in candidates if os.path.exists(path)]

    if not existing:
        raise FileNotFoundError(
            f"Could not find a file for base name '{basename}' in:\n{folder}\n"
            f"Tried: {', '.join(os.path.basename(path) for path in candidates)}"
        )

    priority = {".csv": 0, ".xlsx": 1, ".xls": 2, ".gsheet": 3}
    return min(existing, key=lambda path: priority[os.path.splitext(path)[1].lower()])


def get_gspread_client():
    import gspread
    from google.colab import auth
    from google.auth import default

    auth.authenticate_user()
    credentials, _ = default()
    return gspread.authorize(credentials)


def read_gsheet(path):
    with open(path, encoding="utf-8") as file:
        metadata = json.load(file)

    match = re.search(r"/spreadsheets/d/([a-zA-Z0-9-_]+)", metadata.get("url", ""))
    if not match:
        raise ValueError(f"Could not extract a Google Sheet ID from: {path}")

    worksheet = get_gspread_client().open_by_key(match.group(1)).get_worksheet(0)
    values = worksheet.get_all_values()
    if not values:
        return pd.DataFrame()

    return pd.DataFrame(values[1:], columns=values[0]).replace("", pd.NA)


def read_any_table(path):
    extension = os.path.splitext(path)[1].lower()
    if extension == ".gsheet":
        return read_gsheet(path)
    if extension == ".csv":
        # Try common encodings before falling back to delimiter inference.
        attempts = [
            {"encoding": "utf-8"},
            {"encoding": "utf-8-sig"},
            {"encoding": "cp1252"},
            {"encoding": "latin1"},
            {"encoding": "utf-8", "sep": None, "engine": "python"},
        ]
        last_error = None
        for options in attempts:
            try:
                return pd.read_csv(path, **options)
            except Exception as error:
                last_error = error
        raise last_error
    return pd.read_excel(path)


def to_num(series):
    return pd.to_numeric(series, errors="coerce")


def normalize_text(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip()
    return value or pd.NA


def eng_year_from_root(root):
    parts = str(root).split("/")
    if len(parts) >= 3 and parts[:2] == ["KFUPM", "First Year"]:
        if parts[2] == "First Semester":
            return "First Year/First Semester"
        if parts[2] == "Second Semester":
            return "First Year/Second Semester"
    return parts[1] if len(parts) >= 2 else pd.NA


def eng_major_from_university(university):
    if university == "Taif":
        return "Med"
    if university in {"Taibah", "Jeddah"}:
        return "MechEng"
    if university == "KFUPM":
        return "SWE"
    return pd.NA


def eng_assignment_type(root):
    root = str(root)
    if root.startswith("Jeddah/") or root == "Taibah/First Year":
        return "Graduation Projects"
    if root.startswith("KFUPM/"):
        return "Project Reports"
    if root == "Taif/Medical/First Year/Manuscripts":
        return "Manuscripts"
    if root.startswith("Taif/Medical/First Year/OSCE"):
        return "OSCE"
    if root == "Taif/Medical/Second Year":
        return "Research Projects"
    return pd.NA


def ara_year_from_path(path):
    return str(path).split("/")[0]


def ara_assignment_type(path):
    parts = str(path).split("/")
    if len(parts) >= 2 and parts[0] == "First Year" and parts[1] in {"WT", "PT"}:
        return parts[1]
    return pd.NA


def code_year(semester):
    semester = str(semester).strip()
    if semester == "First Semester":
        return "First Year/First Semester"
    if semester == "Second Semester":
        return "First Year/Second Semester"
    return "First Year"


english_path = find_file_by_basename(DATA_DIR, ENGLISH_BASENAME)
arabic_path = find_file_by_basename(DATA_DIR, ARABIC_BASENAME)
code_path = find_file_by_basename(DATA_DIR, CODE_BASENAME)

print("Found files:")
print(" English:", english_path)
print(" Arabic :", arabic_path)
print(" Code   :", code_path)

eng = read_any_table(english_path).copy()
ara = read_any_table(arabic_path).copy()
cod = read_any_table(code_path).copy()

print("\nShapes:")
print(" English:", eng.shape)
print(" Arabic :", ara.shape)
print(" Code   :", cod.shape)

eng["root"] = eng["root"].astype(str)
eng["Language"] = "English"
eng["University"] = eng["root"].str.split("/").str[0]
eng["Year"] = eng["root"].apply(eng_year_from_root)
eng["Major"] = eng["University"].apply(eng_major_from_university)
eng["Assignment Type"] = eng["root"].apply(eng_assignment_type)

eng_out = pd.DataFrame(
    {
        "Source Dataset": "English",
        "Language": eng["Language"],
        "University": eng["University"],
        "Year": eng["Year"],
        "Major": eng["Major"],
        "Assignment Type": eng["Assignment Type"],
        "file_name": eng["file_name"].map(normalize_text),
        "word_count": to_num(eng["word_count"]),
        "gptzero_ai_rate_percent": to_num(eng["gptzero_ai_rate_percent"]),
        "gptzero_result": eng["gptzero_main_result"].map(normalize_text),
        "pangram_ai_rate_percent": to_num(eng["pangram_ai_rate_percent"]),
        "pangram_result": eng["pangram_main_result"].map(normalize_text),
        "sapling_ai_rate_percent": to_num(eng["sapling_ai_rate_percent"]),
        "sapling_result": eng["sapling_main_result"].map(normalize_text),
        "isgen_ai_rate_percent": to_num(eng["isgen_ai_rate_percent"]),
        "isgen_result": eng["isgen_main_result"].map(normalize_text),
    }
)

ara["relative_path"] = ara["relative_path"].astype(str)
ara["Language"] = "Arabic"
ara["University"] = "Taif"
ara["Year"] = ara["relative_path"].apply(ara_year_from_path)
ara["Major"] = pd.NA
ara["Assignment Type"] = ara["relative_path"].apply(ara_assignment_type)

ara_out = pd.DataFrame(
    {
        "Source Dataset": "Arabic",
        "Language": ara["Language"],
        "University": ara["University"],
        "Year": ara["Year"],
        "Major": ara["Major"],
        "Assignment Type": ara["Assignment Type"],
        "file_name": ara["file_name"].map(normalize_text),
        "word_count": to_num(ara["word_count"]),
        "gptzero_ai_rate_percent": to_num(ara["gptzero_ai_rate_percent"]),
        "gptzero_result": ara["gptzero_result"].map(normalize_text),
        "pangram_ai_rate_percent": to_num(ara["pangram_ai_rate_percent"]),
        "pangram_result": ara["pangram_result"].map(normalize_text),
        "sapling_ai_rate_percent": to_num(ara["sapling_ai_rate_percent"]),
        "sapling_result": ara["sapling_result"].map(normalize_text),
        "isgen_ai_rate_percent": to_num(ara["isgen_ai_rate_percent"]),
        "isgen_result": ara["isgen_result"].map(normalize_text),
    }
)

cod["semester_folder"] = cod["semester_folder"].astype(str)
cod_out = pd.DataFrame(
    {
        "Source Dataset": "Code",
        "Language": "Code",
        "University": "KFUPM",
        "Year": cod["semester_folder"].apply(code_year),
        "Major": "SWE",
        "Assignment Type": "Web Development",
        "file_name": cod["file_name"].map(normalize_text),
        "word_count": to_num(cod["code_word_count"]),
        "gptzero_ai_rate_percent": pd.NA,
        "gptzero_result": pd.NA,
        "pangram_ai_rate_percent": to_num(cod["fraction_ai"]) * 100.0,
        "pangram_result": cod["prediction_short_last_chunk"].map(normalize_text),
        "sapling_ai_rate_percent": pd.NA,
        "sapling_result": pd.NA,
        "isgen_ai_rate_percent": pd.NA,
        "isgen_result": pd.NA,
    }
)

combined = pd.concat([eng_out, ara_out, cod_out], ignore_index=True)

text_columns = [
    "Source Dataset",
    "Language",
    "University",
    "Year",
    "Major",
    "Assignment Type",
    "file_name",
    "gptzero_result",
    "pangram_result",
    "sapling_result",
    "isgen_result",
]
for column in text_columns:
    combined[column] = combined[column].map(normalize_text)

combined = combined.sort_values(
    by=["Source Dataset", "University", "Year", "Assignment Type", "file_name"],
    na_position="last",
).reset_index(drop=True)

combined.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("\nSaved combined file to:")
print(OUTPUT_CSV)
print("\nCombined shape:", combined.shape)
print("\nCounts by source dataset:")
print(combined["Source Dataset"].value_counts(dropna=False))
print("\nCounts by language:")
print(combined["Language"].value_counts(dropna=False))
print("\nPreview:")
with pd.option_context("display.max_columns", None, "display.width", 220):
    print(combined.head(20).to_string(index=False))

## Supporting data check

### Code block 9 — Compute the major distribution for Taif University responses

This block is a supporting data-check cell. It computes the ratio of majors within the Taif subset of the mapped survey data and appears to have been used to contextualize or validate later analyses.

In [ ]:
import pandas as pd
from google.colab import files

UNIVERSITY_COL = "حدّد الجامعة التي تنتمي لها"
MAJOR_COL = r"يرجى تحديد التّخصص\مجال الدراسة"
TARGET_UNIVERSITY = "جامعة الطائف"
OUTPUT_FILE = "taif_major_ratio.csv"

uploaded = files.upload()
input_file = next(iter(uploaded))

# Try common encodings used in exported survey files.
for encoding in ("utf-8", "utf-8-sig", "cp1252"):
    try:
        df = pd.read_csv(input_file, encoding=encoding)
        break
    except UnicodeDecodeError:
        continue
else:
    raise UnicodeDecodeError("unknown", b"", 0, 0, "Unable to decode the uploaded CSV")

df[UNIVERSITY_COL] = df[UNIVERSITY_COL].astype(str).str.strip()
df[MAJOR_COL] = df[MAJOR_COL].astype(str).str.strip()

taif = df.loc[df[UNIVERSITY_COL] == TARGET_UNIVERSITY].copy()

invalid_majors = {"", "nan", "none", "n/a", "na", "<na>"}
taif = taif.loc[~taif[MAJOR_COL].str.lower().isin(invalid_majors)]

counts = (
    taif[MAJOR_COL]
    .value_counts(dropna=False)
    .rename_axis("Major")
    .reset_index(name="Count")
)

total = counts["Count"].sum()
counts["Ratio"] = counts["Count"].astype(str) + ":" + str(total)
counts["Percentage"] = (counts["Count"] / total * 100).round(2)

print("Total Taif responses:", total)
print("\nMajor distribution:")
print(counts.to_string(index=False))

counts.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print("\nSaved:", OUTPUT_FILE)
files.download(OUTPUT_FILE)